In [1]:
from bs4 import BeautifulSoup
import requests

In [69]:
# for index in (1,2):
index=1
url=f"https://www.zameen.com/Flats_Apartments/Lahore-1-{index}.html"
response=requests.get(url)
response.raise_for_status()
print(response.raise_for_status()) # by default if no error then this function return none
# status_code
print(response.status_code)
soup=BeautifulSoup(response.content,"html.parser")
# soup=soup.prettify()
ul_tag=soup.find("ul", class_="e20beb46")

    
    # print("*"*90)
    # print("\n")

None
200


In [70]:
links = []
base_url="https://www.zameen.com"
for a_tag in ul_tag.find_all("a", class_="d870ae17", href=True):
    links.append(base_url + a_tag['href'])

print(links)

['https://www.zameen.com/Property/askari_askari_10_brand_new_13_5_marla_4_bedroom_luxurious_flat_for_sale-53615259-1011-1.html', 'https://www.zameen.com/Property/bahria_town_bahria_town_-_sector_b_with_25_get_your_fully_furnished_one_bed_studio_apartment_with_10_discount_in_sky_luxe_on_2_year_easy_installment-53506631-1772-1.html', 'https://www.zameen.com/Property/askari_11_askari_11_-_sector_b_10_marla_3_bedrooms_apartment_available_for_sale-53612581-15702-1.html', 'https://www.zameen.com/Property/askari_askari_11_brand_new_10_marla_3_bedroom_apartment_available_for_sale_in_askari_11_lahore-53575867-1503-1.html', 'https://www.zameen.com/Property/askari_askari_11_10_marla_3_bedroom_apartment_available_for_sale_in_askari_11_lahore-53612002-1503-1.html', 'https://www.zameen.com/Property/dha_defence_defence_raya_luxury_apartment_for_sale_in_dha_raya_2250_sq_ft_dha_facing-53541671-8172-1.html', 'https://www.zameen.com/Property/gulberg_zahoor_elahi_road_brand_new_2_bed_apartment_for_sale_in

In [71]:
url=links[0]
url

'https://www.zameen.com/Property/askari_askari_10_brand_new_13_5_marla_4_bedroom_luxurious_flat_for_sale-53615259-1011-1.html'

In [72]:
url=links[0]
response=requests.get(url)
response.raise_for_status()
print(response.raise_for_status()) # by default if no error then this function return none
# status_code
print(response.status_code)
soup=BeautifulSoup(response.content,"html.parser")

None
200


In [73]:
def is_area_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) == "Area"

element = soup.find(is_area_li)

In [74]:
# Function to identify the Area li
def is_area_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) == "Area"

# Function to identify the Bedrooms li
def is_bedroom_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) in ["Bedroom(s)", "Beds"]

# Function to identify the Baths li
def is_bath_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) in ["Bath(s)", "Baths"]

area="cb0c0514"

# Extract elements
area_li = soup.find(is_area_li)
bed_li = soup.find(is_bedroom_li)
bath_li = soup.find(is_bath_li)
name_tag = soup.find(class_='cd230541')
price_tag = soup.find(class_='_2923a568')

if area_li and bed_li and bath_li:
    # Bedrooms
    bed_count = bed_li.find('span', class_='_2fdf7fc5').get_text(strip=True)
    
    # Baths
    bath_count = bath_li.find('span', class_='_2fdf7fc5').get_text(strip=True)
    
    # Name with BHK
    name = f"{bed_count} BHK " + (name_tag.text if name_tag else "")
    print("name =", name)
    print("link =", url)

    # Price
    price_raw = price_tag.text.strip() if price_tag else ""
    if price_raw.startswith("PKR") and not price_raw.startswith("PKR "):
        price = "PKR " + price_raw[3:].strip()
    else:
        price = price_raw
    print("price =", price)

    # Area in Marla and sqft
    area_value_span = area_li.find('span', class_='_2fdf7fc5')
    if area_value_span:
        area_text = area_value_span.get_text(strip=True)
        marla_value = float(area_text.split()[0])
        area_sqft = marla_value * 225
        print(f"{marla_value} Marla = {area_sqft:.2f} sq ft")
    else:
        print("Area value span not found")
    
    print("Baths =", bath_count)
    print("Bedroom =", bed_count)
    
else:
    print("Area, Bedroom, or Bath element not found")


name = 4 BHK Askari 10, Askari, Lahore, Punjab
link = https://www.zameen.com/Property/askari_askari_10_brand_new_13_5_marla_4_bedroom_luxurious_flat_for_sale-53615259-1011-1.html
price = PKR 4.42 Crore
13.5 Marla = 3037.50 sq ft
Baths = 5
Bedroom = 4


In [75]:
rooms_section = None
for li in soup.find_all('li', class_='_51519f00'):
    header = li.find('div', class_='_359e9f89')
    if header and header.get_text(strip=True) == "Rooms":
        rooms_section = li
        break

room_details = {}
other_rooms = []

if rooms_section:
    for room_li in rooms_section.find_all('li', class_='_59261156'):
        text = room_li.get_text(strip=True)
        if ':' in text:
            key, value = text.split(':', 1)
            room_details[key.strip()] = value.strip()
        else:
            other_rooms.append(text)

# Print the requested rooms
requested_rooms = ["Study Room", "Servant Quarters", "Drawing Room", "Prayer Room", "Dining Room", "Store Rooms"]
for room in requested_rooms:
    print(f"{room}: {room_details.get(room, 'Not listed')}")

# Print other rooms if any
if other_rooms:
    print("Other Rooms:", ", ".join(other_rooms))

Study Room: Not listed
Servant Quarters: 1
Drawing Room: Not listed
Prayer Room: Not listed
Dining Room: Not listed
Store Rooms: 1
Other Rooms: Drawing Room, Dining Room, Powder Room, Steam Room, Lounge or Sitting Room, Laundry Room


In [76]:
rooms_section = None
for li in soup.find_all('li', class_='_51519f00'):
    header = li.find('div', class_='_359e9f89')
    print(header.get_text(strip=True))
    if header and header.get_text(strip=True) == "Main Features":
        rooms_section = li
        print(rooms_section)
        break

room_details = {}
other_rooms = []

if rooms_section:
    for room_li in rooms_section.find_all('li', class_='_3efd3392'):
        text = room_li.get_text(strip=True)
        print(text)

Main Features
<li class="_51519f00"><div class="_359e9f89"><div class="d0142259">Main Features</div></div><ul class="_3efd3392"><li class="_59261156"><svg class="_0dd20c8b"><use xlink:href="/assets/iconAmenities_noinline.8dbde8780eff18a63210a01d691e1c6b.svg#built-in-year"></use></svg><span class="_9121cbf9">Built in year<!-- -->: 2026</span></li><li class="_59261156"><svg class="_0dd20c8b"><use xlink:href="/assets/iconAmenities_noinline.8dbde8780eff18a63210a01d691e1c6b.svg#parking-spaces"></use></svg><span class="_9121cbf9">Parking Spaces<!-- -->: 2</span></li><li class="_59261156"><svg class="_0dd20c8b"><use xlink:href="/assets/iconAmenities_noinline.8dbde8780eff18a63210a01d691e1c6b.svg#lobby"></use></svg><span class="_9121cbf9">Lobby in Building</span></li><li class="_59261156"><svg class="_0dd20c8b"><use xlink:href="/assets/iconAmenities_noinline.8dbde8780eff18a63210a01d691e1c6b.svg#double-glazed-windows"></use></svg><span class="_9121cbf9">Double Glazed Windows</span></li><li class

In [77]:
address=soup.find(class_="cd230541").text
address

'Askari 10, Askari, Lahore, Punjab'

In [78]:
floor_number = "Not listed"
total_floors = "Not listed"

# Find the "Main Features" section
main_features = None
for li in soup.find_all('li', class_='_51519f00'):
    header = li.find('div', class_='_359e9f89')
    if header and header.get_text(strip=True) == "Main Features":
        main_features = li
        break

if main_features:
    for feature_li in main_features.find_all('li', class_='_59261156'):
        text = feature_li.get_text(strip=True)
        if text.startswith("Floor:"):
            floor_number = text.split(":", 1)[1].strip()
        elif text.startswith("Floors in Building:"):
            total_floors = text.split(":", 1)[1].strip()

print("Floor Number:", floor_number)
print("Total Floors in Building:", total_floors)

Floor Number: 7
Total Floors in Building: 9


In [79]:
built_year = None
for tag in soup.find_all('span', class_='_9121cbf9'):
    text = tag.get_text(strip=True)
    if "Built in year" in text:
        built_year = text.split(":")[-1].strip()
        break

if built_year:
    print("Built in year:", built_year)
else:
    print("Built in year not listed")


Built in year: 2026


In [80]:
features = {}

for li in soup.select("ul._3efd3392 li"):
    span = li.find("span", class_="_9121cbf9")
    if not span:
        continue

    # get full text including value
    full_text = span.get_text(separator=" ", strip=True)

    if ":" in full_text:
        key, value = full_text.split(":", 1)
        features[key.strip()] = value.strip()

print(features)


{'Built in year': '2026', 'Parking Spaces': '2', 'Floor': '7', 'Floors in Building': '9', 'Elevators': '3', 'Bedrooms': '4', 'Bathrooms': '5', 'Servant Quarters': '1', 'Kitchens': '1', 'Store Rooms': '1'}


In [81]:
soup.find(class_="_3547dac9").text

"Brand New 13.5 Marla 4 Bedroom Spacious & Luxurious Flat For Sale!4 Bedrooms with attached washroomsDrawing and Dinning with powder roomKitchen with Brand new CupboardsOpen View with Greenery A Servat quarterAt prime location of Askari 10Walking distance from KFC, MACDONALD'S, CARREFOUR And other outletsNear Beacon House and Bloomfield Hall SchoolGated Community24/7 Security2 mins drive from Airport and Main road"

In [82]:
import re
match = re.search(r'-(\d+)-\d+-\d+\.html$', url)
if match:
    property_id = match.group(1)
    print("Property ID:", property_id)
else:
    print("Property ID not found")

Property ID: 53615259


In [83]:
url

'https://www.zameen.com/Property/askari_askari_10_brand_new_13_5_marla_4_bedroom_luxurious_flat_for_sale-53615259-1011-1.html'

## -----------------------------------

In [84]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import random

base_url = "https://www.zameen.com"
data = []
all_feature_keys = set()

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
})

# Helper functions for listing page (area, beds, baths)
def is_area_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) == "Area"

def is_bedroom_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) in ["Bedroom(s)", "Beds"]

def is_bath_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) in ["Bath(s)", "Baths"]

# ---- MAIN LOOP ----
for i in range(1, 2):
    print(f"Scraping listing page {i}...")
    listing_url = f"{base_url}/Flats_Apartments/Lahore-1-{i}.html"

    try:
        response = session.get(listing_url, timeout=20)
        response.raise_for_status()
    except Exception as e:
        print(f"Error loading listing page {i}: {e}")
        continue

    soup = BeautifulSoup(response.content, "html.parser")

    # Grab all property links
    ul_tag = soup.find("ul", class_="e20beb46")
    links = []
    if ul_tag:
        for a_tag in ul_tag.find_all("a", class_="d870ae17", href=True):
            href = a_tag["href"]
            if href.startswith("/"):
                links.append(base_url + href)
            else:
                links.append(href)

    # ---- PROPERTY LOOP ----
    for url in links:
        try:
            prop_res = session.get(url, timeout=20)
            prop_res.raise_for_status()
            soup = BeautifulSoup(prop_res.content, "html.parser")

            # Title/Name + Price
            name_tag = soup.find(class_='cd230541')
            price_tag = soup.find(class_='_2923a568')

            # Area/Beds/Baths (from overview list)
            area_li = soup.find(is_area_li)
            bed_li = soup.find(is_bedroom_li)
            bath_li = soup.find(is_bath_li)

            bed_count = bath_count = marla_value = area_sqft = None
            name = price = ""

            if bed_li:
                val = bed_li.find('span', class_='_2fdf7fc5')
                bed_count = val.get_text(strip=True) if val else None

            if bath_li:
                val = bath_li.find('span', class_='_2fdf7fc5')
                bath_count = val.get_text(strip=True) if val else None

            if name_tag:
                # If bed_count exists, prefix like "3 BHK ..."
                if bed_count:
                    name = f"{bed_count} BHK {name_tag.get_text(strip=True)}"
                else:
                    name = name_tag.get_text(strip=True)

            price_raw = price_tag.get_text(strip=True) if price_tag else ""
            if price_raw.startswith("PKR") and not price_raw.startswith("PKR "):
                price = "PKR " + price_raw[3:].strip()
            else:
                price = price_raw

            # Area conversion (marla -> sqft)
            if area_li:
                area_value_span = area_li.find('span', class_='_2fdf7fc5')
                if area_value_span:
                    area_text = area_value_span.get_text(strip=True)
                    # ex: "10 Marla" -> 10
                    try:
                        marla_value = float(area_text.split()[0])
                        area_sqft = marla_value * 225
                    except:
                        marla_value = None
                        area_sqft = None

            # Address (your earlier code used name_tag; using safer selector if available)
            address = "Not listed"
            if name_tag:
                address = name_tag.get_text(strip=True)

            # ---- MAIN FEATURES (DYNAMIC -> COLUMNS) ----
            # This extracts:
            # Built in year, Parking Spaces, Floors in Building, Bedrooms, Bathrooms, Servant Quarters, Kitchens, Store Rooms, etc.
            main_features_dict = {}
            for li in soup.select("ul._3efd3392 li"):
                span = li.find("span", class_="_9121cbf9")
                if not span:
                    continue
                full_text = span.get_text(separator=" ", strip=True)

                if ":" in full_text:
                    key, value = full_text.split(":", 1)
                    key = key.strip()
                    value = value.strip()
                    if key:
                        main_features_dict[key] = value

            all_feature_keys.update(main_features_dict.keys())

            # These are your specific fields mapped from features
            built_year = main_features_dict.get("Built in year")
            parking_spaces = main_features_dict.get("Parking Spaces")

            # Some listings may label floor differently; keep both safe
            floor_number = main_features_dict.get("Floor") or main_features_dict.get("Floor Number")
            total_floors = main_features_dict.get("Floors in Building")

            # ---- ROOMS SECTION (your existing logic) ----
            rooms_section = None
            for li in soup.find_all('li', class_='_51519f00'):
                header = li.find('div', class_='_359e9f89')
                if header and header.get_text(strip=True) == "Rooms":
                    rooms_section = li
                    break

            room_details = {}
            other_rooms = []
            if rooms_section:
                for room_li in rooms_section.find_all('li', class_='_59261156'):
                    text = room_li.get_text(separator=" ", strip=True)
                    if ':' in text:
                        key, value = text.split(':', 1)
                        room_details[key.strip()] = value.strip()
                    else:
                        other_rooms.append(text)

            # Property ID from URL
            match = re.search(r'-(\d+)-\d+-\d+\.html$', url)
            property_id = match.group(1) if match else "Not found"

            # ---- BUILD ROW ----
            row = {
                "Property ID": property_id,
                "Name": name,
                "Price": price,
                "Marla": marla_value,
                "Area (sqft)": area_sqft,
                "Bedrooms": bed_count,
                "Baths": bath_count,
                "Floor Number": floor_number,
                "Total Floors": total_floors,
                "Built Year": built_year,
                "Parking Spaces": parking_spaces,
                "Address": address,
                "Rooms": room_details,
                "Other Rooms": ", ".join(other_rooms),
                "Link": url
            }

            # Add ALL dynamic main feature keys as their own columns
            for k, v in main_features_dict.items():
                row[k] = v

            data.append(row)

            # polite delay (avoid blocks)
            time.sleep(random.uniform(0.5, 1.2))

        except Exception as e:
            print(f"Error scraping {url}: {e}")

# ---- FINAL DATAFRAME ----
df = pd.DataFrame(data)

# Ensure all feature columns exist (even if missing for some rows)
for k in all_feature_keys:
    if k not in df.columns:
        df[k] = None

print(df.head())


Scraping listing page 1...
  Property ID                                               Name  \
0    53615259            4 BHK Askari 10, Askari, Lahore, Punjab   
1    53506631  - BHK Bahria Town - Sector B, Bahria Town, Lah...   
2    53612581  3 BHK Askari 11 - Sector B, Askari 11, Askari,...   
3    53575867            3 BHK Askari 11, Askari, Lahore, Punjab   
4    53612002            3 BHK Askari 11, Askari, Lahore, Punjab   

            Price  Marla  Area (sqft) Bedrooms Baths Floor Number  \
0  PKR 4.42 Crore   13.5       3037.5        4     5            7   
1  PKR 21.57 Lakh    1.8        405.0        1     1         None   
2  PKR 3.25 Crore   10.0       2250.0        3     4         None   
3   PKR 3.9 Crore   10.0       2250.0        3     4         None   
4  PKR 3.25 Crore   10.0       2250.0        3     4         None   

  Total Floors Built Year  ...  \
0            9       2026  ...   
1           13       None  ...   
2         None       2022  ...   
3         Non

In [4]:
df.iloc[0]

Property ID                                                    53608085
Name                            3 BHK Askari 11, Askari, Lahore, Punjab
Price                                                    PKR 2.85 Crore
Marla                                                              10.0
Area (sqft)                                                      2250.0
Bedrooms                                                              3
Baths                                                                 4
Floor Number                                                       None
Total Floors                                                          8
Built Year                                                         2018
Parking Spaces                                                        2
Address                               Askari 11, Askari, Lahore, Punjab
Rooms                 {'Bedrooms': '3', 'Bathrooms': '4', 'Servant Q...
Other Rooms           Drawing Room, Dining Room, Study Room, Pra

In [72]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

base_url = "https://www.zameen.com"
data = []  # store all listings here

# Loop through first 50 pages
for i in range(1, 51):
    print(f"Scraping listing page {i}...")
    listing_url = f"https://www.zameen.com/Flats_Apartments/Lahore-1-{i}.html"
    response = requests.get(listing_url)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")

    ul_tag = soup.find("ul", class_="e20beb46")
    links = []
    if ul_tag:
        for a_tag in ul_tag.find_all("a", class_="d870ae17", href=True):
            links.append(base_url + a_tag['href'])

    # Helper functions
    def is_area_li(tag):
        if tag.name != 'li':
            return False
        label_span = tag.find('span', class_='ed0db22a')
        return label_span and label_span.get_text(strip=True) == "Area"

    def is_bedroom_li(tag):
        if tag.name != 'li':
            return False
        label_span = tag.find('span', class_='ed0db22a')
        return label_span and label_span.get_text(strip=True) in ["Bedroom(s)", "Beds"]

    def is_bath_li(tag):
        if tag.name != 'li':
            return False
        label_span = tag.find('span', class_='ed0db22a')
        return label_span and label_span.get_text(strip=True) in ["Bath(s)", "Baths"]

    # Loop through each property link
    for url in links:
        try:
            prop_res = requests.get(url)
            prop_res.raise_for_status()
            soup = BeautifulSoup(prop_res.content, "html.parser")

            name_tag = soup.find(class_='cd230541')
            price_tag = soup.find(class_='_2923a568')
            area_li = soup.find(is_area_li)
            bed_li = soup.find(is_bedroom_li)
            bath_li = soup.find(is_bath_li)

            bed_count = bath_count = marla_value = area_sqft = None
            name = price = ""

            if area_li and bed_li and bath_li:
                bed_count = bed_li.find('span', class_='_2fdf7fc5').get_text(strip=True)
                bath_count = bath_li.find('span', class_='_2fdf7fc5').get_text(strip=True)
                name = f"{bed_count} BHK " + (name_tag.text if name_tag else "")
                price_raw = price_tag.text.strip() if price_tag else ""
                if price_raw.startswith("PKR") and not price_raw.startswith("PKR "):
                    price = "PKR " + price_raw[3:].strip()
                else:
                    price = price_raw
                area_value_span = area_li.find('span', class_='_2fdf7fc5')
                if area_value_span:
                    area_text = area_value_span.get_text(strip=True)
                    marla_value = float(area_text.split()[0])
                    area_sqft = marla_value * 225

            # Address
            address = name_tag.text if name_tag else "Not listed"

            # Floor info
            floor_number = total_floors = "Not listed"
            main_features = None
            for li in soup.find_all('li', class_='_51519f00'):
                header = li.find('div', class_='_359e9f89')
                if header and header.get_text(strip=True) == "Main Features":
                    main_features = li
                    break
            if main_features:
                for feature_li in main_features.find_all('li', class_='_59261156'):
                    text = feature_li.get_text(strip=True)
                    if text.startswith("Floor:"):
                        floor_number = text.split(":", 1)[1].strip()
                    elif text.startswith("Floors in Building:"):
                        total_floors = text.split(":", 1)[1].strip()

            # Built year
            built_year = None
            for tag in soup.find_all('span', class_='_9121cbf9'):
                text = tag.get_text(strip=True)
                if "Built in year" in text:
                    built_year = text.split(":")[-1].strip()
                    break

            # Rooms
            rooms_section = None
            for li in soup.find_all('li', class_='_51519f00'):
                header = li.find('div', class_='_359e9f89')
                if header and header.get_text(strip=True) == "Rooms":
                    rooms_section = li
                    break
            room_details = {}
            other_rooms = []
            if rooms_section:
                for room_li in rooms_section.find_all('li', class_='_59261156'):
                    text = room_li.get_text(strip=True)
                    if ':' in text:
                        key, value = text.split(':', 1)
                        room_details[key.strip()] = value.strip()
                    else:
                        other_rooms.append(text)

            # Property ID
            match = re.search(r'-(\d+)-\d+-\d+\.html$', url)
            property_id = match.group(1) if match else "Not found"

            # Save data
            data.append({
                "Property ID": property_id,
                "Name": name,
                "Price": price,
                "Marla": marla_value,
                "Area (sqft)": area_sqft,
                "Bedrooms": bed_count,
                "Baths": bath_count,
                "Floor Number": floor_number,
                "Total Floors": total_floors,
                "Built Year": built_year,
                "Address": address,
                "Rooms": room_details,
                "Other Rooms": ", ".join(other_rooms),
                "Link": url
            })
        except Exception as e:
            print(f"Error scraping {url}: {e}")

# Final DataFrame
df = pd.DataFrame(data)
df.head()


Scraping listing page 1...
Scraping listing page 2...
Scraping listing page 3...
Scraping listing page 4...
Scraping listing page 5...
Scraping listing page 6...
Scraping listing page 7...
Scraping listing page 8...
Scraping listing page 9...
Scraping listing page 10...
Scraping listing page 11...
Scraping listing page 12...
Scraping listing page 13...
Scraping listing page 14...
Scraping listing page 15...
Scraping listing page 16...
Scraping listing page 17...
Scraping listing page 18...
Scraping listing page 19...
Scraping listing page 20...
Scraping listing page 21...
Scraping listing page 22...
Scraping listing page 23...
Scraping listing page 24...
Scraping listing page 25...
Scraping listing page 26...
Scraping listing page 27...
Scraping listing page 28...
Scraping listing page 29...
Scraping listing page 30...
Scraping listing page 31...
Scraping listing page 32...
Scraping listing page 33...
Scraping listing page 34...
Scraping listing page 35...
Scraping listing page 36...
S

,Property ID,Name,Price,Marla,Area (sqft),Bedrooms,Baths,Floor Number,Total Floors,Built Year,Address,Rooms,Other Rooms,Link
0,50175391,"1 BHK Bahria Town - Sector E, Bahria Town, Lah...",PKR 65 Lakh,2.2,495.0,1,1,Not listed,22,2022,"Bahria Town - Sector E, Bahria Town, Lahore, P...","{'Bedrooms': '5', 'Bathrooms': '6', 'Servant Q...","Drawing Room, Dining Room, Study Room, Prayer ...",https://www.zameen.com/Property/bahria_town_ba...
1,50382634,"1 BHK Bahria Town - Tauheed Block, Bahria Town...",PKR 58 Lakh,2.0,450.0,1,1,Not listed,7,2023,"Bahria Town - Tauheed Block, Bahria Town - Sec...","{'Bedrooms': '1', 'Bathrooms': '1', 'Servant Q...","Drawing Room, Dining Room, Kitchens, Study Roo...",https://www.zameen.com/Property/bahria_town_se...
2,52897716,"3 BHK Askari 11 - Sector D, Askari 11, Askari,...",PKR 4.05 Crore,10.9,2452.5,3,4,2,20,2025,"Askari 11 - Sector D, Askari 11, Askari, Lahor...","{'Bedrooms': '3', 'Bathrooms': '4', 'Servant Q...","Drawing Room, Dining Room, Study Room, Prayer ...",https://www.zameen.com/Property/askari_11_aska...
3,52566838,"1 BHK Midway Commercial, Bahria Town, Lahore, ...",PKR 29 Lakh,1.8,405.0,1,1,Not listed,14,2022,"Midway Commercial, Bahria Town, Lahore, Punjab","{'Bedrooms': '1', 'Bathrooms': '1', 'Kitchens'...","Servant Quarters, Drawing Room, Dining Room, S...",https://www.zameen.com/Property/bahria_town_mi...
4,52380886,"1 BHK Bahria Town - Johar Block, Bahria Town -...",PKR 93.5 Lakh,2.4,540.0,1,1,8,11,None,"Bahria Town - Johar Block, Bahria Town - Secto...","{'Bedrooms': '1', 'Bathrooms': '1', 'Kitchens'...",,https://www.zameen.com/Property/bahria_town_se...


In [73]:
df.shape

(1282, 14)

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from urllib.parse import urljoin

# Define all URLs with their page counts
url_configs = [
    # URLs with multiple pages
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Askari-3555-{index}.html", "pages": 37},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Town-509-{index}.html", "pages": 22},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Gulberg-7-{index}.html", "pages": 22},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Raiwind_Road-128-{index}.html", "pages": 12},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_DHA_Defence-9-{index}.html", "pages": 11},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Orchard-1474-{index}.html", "pages": 4}, 
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Main_Canal_Bank_Road-431-{index}.html", "pages": 5},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Johar_Town-93-{index}.html", "pages": 3},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Land_Breeze_Housing_Society-9784-{index}.html", "pages": 2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Lakshmi_Chowk-1505-{index}.html", "pages": 2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Shah_Jamal-1491-{index}.html", "pages": 1},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Defence_Road-4574-{index}.html", "pages": 2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Ferozepur_Road-81-{index}.html", "pages": 1},
]

# Single page URLs
single_page_urls = [
    "https://www.zameen.com/Flats_Apartments/Lahore_Khayaban_e_Amin-1514-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Eden-3098-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shanghai_Road-12334-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Wapda_Town-423-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Garden_Town-4-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Jubilee_Town-766-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Khayaban_e_Jinnah_Road-8145-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Cooper_Road-12238-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Allama_Iqbal_Town-58-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Central_Park_Housing_Scheme-1013-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Sukh_Chayn_Gardens-473-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Nasheman-4510-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shadman-15803-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Jail_Road-528-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Ichhra-88-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Lahore_Villas-12049-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_New_Lahore_City-8152-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Park_Avenue_Housing_Scheme-10934-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Izmir_Town-483-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Model_Town-8-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Canal_Garden-1287-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Khayaban_e_Zafar-18840-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Multan_Road-48-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Kamahan_Road-8855-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Ring_Road-4575-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Lawrence_Road-8288-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Chinar_Court-1462-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_GT_Road-472-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Qartaba_Chowk-12099-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_State_Life_Housing_Society-487-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Kacha_Ferozepur_Road-11330-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Rehman_Gardens-4584-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Labor_Colony-10932-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Paragon_City-1014-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_IEP_Engineers_Town-8425-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Fazaia_Housing_Scheme-1677-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Thokar_Niaz_Baig-145-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Audit__amp__Accounts_Co_operative_Society-1134-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Valencia_Housing_Society-373-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Maulana_Shaukat_Ali_Road-9866-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Icon_Valley-17703-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_LDA_Avenue-547-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Noor_Jahan_Road-10669-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Davis_Road-387-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shama_Road-12148-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Sabzazar_Scheme-562-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Ravi_Gardens_Smart_Homes-21814-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Lytton_Road-4581-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Punjab_Co_operative_Housing_Society-456-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Education___Medical_City-8334-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Cantt-427-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shabbir_Town-4389-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Islam_Pura-1369-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Samanabad-377-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Chauburji-13907-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Abdul_Sattar_Edhi_Road-10858-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Mustafa_Town-325-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Medical_Housing_Society-4359-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Faisal_Town-79-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shahkam_Chowk-11216-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_DHA_11_Rahbar-1410-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Revenue_Society-1359-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Kotha_Pind-10955-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Wafaqi_Colony-4380-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Walton_Road-153-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Punjab_Govt_Employees_Society-4430-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_LDA_Road-13991-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Vital_Homes_Housing_Scheme-9461-1.html",
    
]

base_url = "https://www.zameen.com"
data = []  # store all listings here

# Helper functions
def is_area_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) == "Area"

def is_bedroom_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) in ["Bedroom(s)", "Beds"]

def is_bath_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) in ["Bath(s)", "Baths"]

def scrape_property_links(listing_url):
    """Extract property links from a listing page"""
    try:
        response = requests.get(listing_url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")
        
        ul_tag = soup.find("ul", class_="e20beb46")
        links = []
        if ul_tag:
            for a_tag in ul_tag.find_all("a", class_="d870ae17", href=True):
                links.append(base_url + a_tag['href'])
        return links
    except Exception as e:
        print(f"Error scraping listing page {listing_url}: {e}")
        return []

def scrape_property_details(url):
    """Scrape detailed information from a property page"""
    try:
        prop_res = requests.get(url)
        prop_res.raise_for_status()
        soup = BeautifulSoup(prop_res.content, "html.parser")

        name_tag = soup.find(class_='cd230541')
        price_tag = soup.find(class_='_2923a568')
        area_li = soup.find(is_area_li)
        bed_li = soup.find(is_bedroom_li)
        bath_li = soup.find(is_bath_li)

        bed_count = bath_count = marla_value = area_sqft = None
        name = price = ""

        if area_li and bed_li and bath_li:
            bed_count = bed_li.find('span', class_='_2fdf7fc5').get_text(strip=True)
            bath_count = bath_li.find('span', class_='_2fdf7fc5').get_text(strip=True)
            name = f"{bed_count} BHK " + (name_tag.text if name_tag else "")
            price_raw = price_tag.text.strip() if price_tag else ""
            if price_raw.startswith("PKR") and not price_raw.startswith("PKR "):
                price = "PKR " + price_raw[3:].strip()
            else:
                price = price_raw
            area_value_span = area_li.find('span', class_='_2fdf7fc5')
            if area_value_span:
                area_text = area_value_span.get_text(strip=True)
                try:
                    marla_value = float(area_text.split()[0])
                    area_sqft = marla_value * 225
                except (ValueError, IndexError):
                    pass

        # Address
        address = name_tag.text if name_tag else "Not listed"

        # Floor info
        floor_number = total_floors = "Not listed"
        main_features = None
        for li in soup.find_all('li', class_='_51519f00'):
            header = li.find('div', class_='_359e9f89')
            if header and header.get_text(strip=True) == "Main Features":
                main_features = li
                break
        if main_features:
            for feature_li in main_features.find_all('li', class_='_59261156'):
                text = feature_li.get_text(strip=True)
                if text.startswith("Floor:"):
                    floor_number = text.split(":", 1)[1].strip()
                elif text.startswith("Floors in Building:"):
                    total_floors = text.split(":", 1)[1].strip()

        # Built year
        built_year = None
        for tag in soup.find_all('span', class_='_9121cbf9'):
            text = tag.get_text(strip=True)
            if "Built in year" in text:
                built_year = text.split(":")[-1].strip()
                break

        # Rooms
        rooms_section = None
        for li in soup.find_all('li', class_='_51519f00'):
            header = li.find('div', class_='_359e9f89')
            if header and header.get_text(strip=True) == "Rooms":
                rooms_section = li
                break
        room_details = {}
        other_rooms = []
        if rooms_section:
            for room_li in rooms_section.find_all('li', class_='_59261156'):
                text = room_li.get_text(strip=True)
                if ':' in text:
                    key, value = text.split(':', 1)
                    room_details[key.strip()] = value.strip()
                else:
                    other_rooms.append(text)

        # Property ID
        match = re.search(r'-(\d+)-\d+-\d+\.html$', url)
        property_id = match.group(1) if match else "Not found"

        return {
            "Property ID": property_id,
            "Name": name,
            "Price": price,
            "Marla": marla_value,
            "Area (sqft)": area_sqft,
            "Bedrooms": bed_count,
            "Baths": bath_count,
            "Floor Number": floor_number,
            "Total Floors": total_floors,
            "Built Year": built_year,
            "Address": address,
            "Rooms": room_details,
            "Other Rooms": ", ".join(other_rooms),
            "Link": url
        }
    except Exception as e:
        print(f"Error scraping property {url}: {e}")
        return None

# Process URLs with multiple pages
print("Starting to scrape multi-page URLs...")
total_configs = len(url_configs)
for config_idx, config in enumerate(url_configs, 1):
    base_template = config["base_url"]
    pages = config["pages"]
    location_name = base_template.split('/')[-1].split('-')[0].replace('_', ' ')
    
    print(f"[{config_idx}/{total_configs}] Processing {location_name} - {pages} pages...")
    
    # Loop through all pages for this location
    for page in range(1, pages + 1):
        listing_url = base_template.format(index=page)
        print(f"  Scraping page {page}/{pages} for {location_name}...")
        
        # Get property links from this listing page
        property_links = scrape_property_links(listing_url)
        print(f"  Found {len(property_links)} properties on page {page}")
        
        # Scrape each property
        for link_idx, prop_url in enumerate(property_links, 1):
            property_data = scrape_property_details(prop_url)
            if property_data:
                data.append(property_data)
            
            # Add small delay to be respectful
            if link_idx % 10 == 0:
                print(f"    Scraped {link_idx}/{len(property_links)} properties from page {page}")
                time.sleep(1)
        
        # Small delay between pages
        time.sleep(0.5)

# Process single page URLs
print(f"\nStarting to scrape {len(single_page_urls)} single-page URLs...")
for url_idx, listing_url in enumerate(single_page_urls, 1):
    location_name = listing_url.split('/')[-1].split('-')[0].replace('_', ' ')
    print(f"[{url_idx}/{len(single_page_urls)}] Processing {location_name}...")
    
    # Get property links from this listing page
    property_links = scrape_property_links(listing_url)
    print(f"  Found {len(property_links)} properties")
    
    # Scrape each property
    for link_idx, prop_url in enumerate(property_links, 1):
        property_data = scrape_property_details(prop_url)
        if property_data:
            data.append(property_data)
        
        # Add small delay to be respectful
        if link_idx % 10 == 0:
            print(f"    Scraped {link_idx}/{len(property_links)} properties")
            time.sleep(1)
    
    # Small delay between URLs
    time.sleep(0.5)

# Create final DataFrame
print(f"\nScraping completed! Total properties collected: {len(data)}")
df = pd.DataFrame(data)

# Display summary statistics
print("\nDataFrame Info:")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print("\nColumn names:")
for col in df.columns:
    print(f"  - {col}")

# Display first few rows
print("\nFirst 5 rows:")
print(df.head())

# Save to CSV
csv_filename = "lahore_flats.csv"
# df.to_csv(csv_filename, index=False)
print(f"\nData saved to {csv_filename}")

# Display some basic statistics
if not df.empty:
    print("\nBasic Statistics:")
    print(f"Price range: {df['Price'].nunique()} unique prices")
    print(f"Bedroom types: {df['Bedrooms'].nunique()} different bedroom counts")
    print(f"Areas covered: {df['Address'].nunique()} unique addresses")
    print(f"Average Marla: {df['Marla'].mean():.2f}" if df['Marla'].notna().any() else "No Marla data")
    print(f"Average Area (sqft): {df['Area (sqft)'].mean():.2f}" if df['Area (sqft)'].notna().any() else "No Area data")

print("\nScraping process completed successfully!")

Starting to scrape multi-page URLs...
[1/13] Processing Lahore Bahria Town - 32 pages...
  Scraping page 1/32 for Lahore Bahria Town...
  Found 25 properties on page 1
    Scraped 10/25 properties from page 1
    Scraped 20/25 properties from page 1
  Scraping page 2/32 for Lahore Bahria Town...
  Found 25 properties on page 2
    Scraped 10/25 properties from page 2
    Scraped 20/25 properties from page 2
  Scraping page 3/32 for Lahore Bahria Town...
  Found 25 properties on page 3
    Scraped 10/25 properties from page 3
    Scraped 20/25 properties from page 3
  Scraping page 4/32 for Lahore Bahria Town...
  Found 25 properties on page 4
    Scraped 10/25 properties from page 4
    Scraped 20/25 properties from page 4
  Scraping page 5/32 for Lahore Bahria Town...
  Found 25 properties on page 5
    Scraped 10/25 properties from page 5
    Scraped 20/25 properties from page 5
  Scraping page 6/32 for Lahore Bahria Town...
  Found 25 properties on page 6
    Scraped 10/25 propertie

In [4]:
df.shape

(3626, 14)

In [5]:
df.to_csv(csv_filename, index=False)

In [63]:
# Define all URLs with their page counts
url_configs = [
    # URLs with multiple pages
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Town-509-{index}.html", "pages": 32},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Askari-3555-{index}.html", "pages": 31},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Gulberg-7-{index}.html", "pages": 20},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Raiwind_Road-128-{index}.html", "pages": 12},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_DHA_Defence-9-{index}.html", "pages": 8},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Orchard-1474-{index}.html", "pages": 5},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Main_Canal_Bank_Road-431-{index}.html", "pages": 4},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Johar_Town-93-{index}.html", "pages": 3},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Land_Breeze_Housing_Society-9784-{index}.html", "pages": 2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Lakshmi_Chowk-1505-{index}.html", "pages": 2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Shah_Jamal-1491-{index}.html", "pages": 2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Defence_Road-4574-{index}.html", "pages": 2},
    {"base_url":"https://www.zameen.com/Flats_Apartments/Lahore_Shanghai_Road-12334-1.html","pages":2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Ferozepur_Road-81-{index}.html", "pages": 1},
]

# Single page URLs
single_page_urls = [
    "https://www.zameen.com/Flats_Apartments/Lahore_Khayaban_e_Amin-1514-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Eden-3098-1.html",
    
    "https://www.zameen.com/Flats_Apartments/Lahore_Wapda_Town-423-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Garden_Town-4-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Jubilee_Town-766-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Khayaban_e_Jinnah_Road-8145-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Cooper_Road-12238-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Allama_Iqbal_Town-58-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Central_Park_Housing_Scheme-1013-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Sukh_Chayn_Gardens-473-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Nasheman-4510-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shadman-15803-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Jail_Road-528-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Ichhra-88-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Lahore_Villas-12049-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_New_Lahore_City-8152-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Park_Avenue_Housing_Scheme-10934-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Izmir_Town-483-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Model_Town-8-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Canal_Garden-1287-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Khayaban_e_Zafar-18840-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Multan_Road-48-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Kamahan_Road-8855-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Ring_Road-4575-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Lawrence_Road-8288-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_GT_Road-472-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Qartaba_Chowk-12099-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_State_Life_Housing_Society-487-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Kacha_Ferozepur_Road-11330-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Rehman_Gardens-4584-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Labor_Colony-10932-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Paragon_City-1014-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_IEP_Engineers_Town-8425-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Fazaia_Housing_Scheme-1677-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Thokar_Niaz_Baig-145-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Valencia_Housing_Society-373-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Maulana_Shaukat_Ali_Road-9866-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Icon_Valley-17703-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_LDA_Avenue-547-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Noor_Jahan_Road-10669-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Davis_Road-387-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shama_Road-12148-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Sabzazar_Scheme-562-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Ravi_Gardens_Smart_Homes-21814-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Lytton_Road-4581-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Punjab_Co_operative_Housing_Society-456-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Education___Medical_City-8334-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Cantt-427-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shabbir_Town-4389-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Islam_Pura-1369-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Samanabad-377-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Chauburji-13907-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Abdul_Sattar_Edhi_Road-10858-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Mustafa_Town-325-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Medical_Housing_Society-4359-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Faisal_Town-79-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shahkam_Chowk-11216-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_DHA_11_Rahbar-1410-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Revenue_Society-1359-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Kotha_Pind-10955-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Wafaqi_Colony-4380-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Walton_Road-153-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Punjab_Govt_Employees_Society-4430-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_LDA_Road-13991-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Pine_Avenue-14395-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_PIA_Housing_Scheme-444-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Iqbal_Avenue-1269-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Rehmanpura__Ferozpur_Road_-1683-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Mozang-8117-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Zaman_Colony-4472-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Band_Road-1212-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Pico_Road-21483-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shalimar_Link_Road-9437-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Muslim_Town-326-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Sant_Nagar-4537-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Super_Town-8504-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Saadi_Park-4483-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Waheed_Brother_Colony-4494-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_New_Garden_Town-16690-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_EME_Society-410-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Vital_Homes_Housing_Scheme-9461-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Cavalry_Ground-69-1.html"
    ""
]

base_url = "https://www.zameen.com"
data = []  # store all listings here


import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

base_url = "https://www.zameen.com"
data = []
all_feature_keys = set()   # ✅ track all dynamic feature keys

# -------------------------
# Helper functions
# -------------------------
def is_area_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) == "Area"

def is_bedroom_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) in ["Bedroom(s)", "Beds"]

def is_bath_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) in ["Bath(s)", "Baths"]

def scrape_property_links(listing_url: str):
    soup = get_soup(listing_url)

    links = []

    # --- Method 1 (your original) ---
    ul = soup.find("ul", class_="e20beb46")
    if ul:
        for a in ul.find_all("a", class_="d870ae17", href=True):
            links.append(urljoin(BASE, a["href"]))

    # --- Method 2 (fallback: any property link pattern) ---
    if not links:
        for a in soup.find_all("a", href=True):
            href = a["href"]
            # property pages usually contain "/Property/"
            if "/Property/" in href:
                links.append(urljoin(BASE, href))

    # unique
    return list(dict.fromkeys(links))


def scrape_property_details(url):
    """Scrape detailed information from a property page"""
    global all_feature_keys

    try:
        prop_res = requests.get(url, timeout=20)
        prop_res.raise_for_status()
        soup = BeautifulSoup(prop_res.content, "html.parser")

        name_tag = soup.find(class_='cd230541')
        price_tag = soup.find(class_='_2923a568')
        area_li = soup.find(is_area_li)
        bed_li = soup.find(is_bedroom_li)
        bath_li = soup.find(is_bath_li)

        bed_count = bath_count = marla_value = area_sqft = None
        name = price = ""

        # ---- Beds/Baths ----
        if bed_li:
            bed_span = bed_li.find('span', class_='_2fdf7fc5')
            bed_count = bed_span.get_text(strip=True) if bed_span else None

        if bath_li:
            bath_span = bath_li.find('span', class_='_2fdf7fc5')
            bath_count = bath_span.get_text(strip=True) if bath_span else None

        # ---- Name ----
        if name_tag:
            if bed_count:
                name = f"{bed_count} BHK {name_tag.get_text(strip=True)}"
            else:
                name = name_tag.get_text(strip=True)

        # ---- Price ----
        price_raw = price_tag.get_text(strip=True) if price_tag else ""
        if price_raw.startswith("PKR") and not price_raw.startswith("PKR "):
            price = "PKR " + price_raw[3:].strip()
        else:
            price = price_raw

        # ---- Area (Marla -> sqft) ----
        if area_li:
            area_value_span = area_li.find('span', class_='_2fdf7fc5')
            if area_value_span:
                area_text = area_value_span.get_text(strip=True)
                try:
                    marla_value = float(area_text.split()[0])
                    area_sqft = marla_value * 225
                except (ValueError, IndexError):
                    marla_value = None
                    area_sqft = None

        # ---- Address ----
        address = name_tag.get_text(strip=True) if name_tag else "Not listed"

        # ------------------------------------------------------
        # ✅ MAIN FEATURES (extract ALL features automatically)
        # ------------------------------------------------------
        main_features_dict = {}

        # This UL contains things like:
        # Built in year, Parking Spaces, Floors in Building, Bedrooms, Bathrooms, Kitchens, etc.
        for li in soup.select("ul._3efd3392 li"):
            span = li.find("span", class_="_9121cbf9")
            if not span:
                continue

            full_text = span.get_text(separator=" ", strip=True)
            if ":" in full_text:
                key, value = full_text.split(":", 1)
                key = key.strip()
                value = value.strip()
                if key:
                    main_features_dict[key] = value

        all_feature_keys.update(main_features_dict.keys())

        # optional direct mapped fields (if you still want these fixed columns)
        floor_number = main_features_dict.get("Floor") or main_features_dict.get("Floor Number")
        total_floors = main_features_dict.get("Floors in Building")
        built_year = main_features_dict.get("Built in year")

        # ------------------------------------------------------
        # Rooms section (your existing logic)
        # ------------------------------------------------------
        rooms_section = None
        for li in soup.find_all('li', class_='_51519f00'):
            header = li.find('div', class_='_359e9f89')
            if header and header.get_text(strip=True) == "Rooms":
                rooms_section = li
                break

        room_details = {}
        other_rooms = []
        if rooms_section:
            for room_li in rooms_section.find_all('li', class_='_59261156'):
                text = room_li.get_text(separator=" ", strip=True)
                if ':' in text:
                    key, value = text.split(':', 1)
                    room_details[key.strip()] = value.strip()
                else:
                    other_rooms.append(text)

        # Property ID
        match = re.search(r'-(\d+)-\d+-\d+\.html$', url)
        property_id = match.group(1) if match else "Not found"

        # ------------------------------------------------------
        # Build row
        # ------------------------------------------------------
        row = {
            "Property ID": property_id,
            "Name": name,
            "Price": price,
            "Marla": marla_value,
            "Area (sqft)": area_sqft,
            "Bedrooms": bed_count,
            "Baths": bath_count,
            "Floor Number": floor_number,
            "Total Floors": total_floors,
            "Built Year": built_year,
            "Address": address,
            "Rooms": room_details,
            "Other Rooms": ", ".join(other_rooms),
            "Link": url
        }

        # ✅ Add every main feature as a separate column
        for k, v in main_features_dict.items():
            row[k] = v

        return row

    except Exception as e:
        print(f"Error scraping property {url}: {e}")
        return None

# -------------------------
# Process URLs with multiple pages
# -------------------------
print("Starting to scrape multi-page URLs...")
total_configs = len(url_configs)

for config_idx, config in enumerate(url_configs, 1):
    base_template = config["base_url"]
    pages = config["pages"]
    location_name = base_template.split('/')[-1].split('-')[0].replace('_', ' ')

    print(f"[{config_idx}/{total_configs}] Processing {location_name} - {pages} pages...")

    for page in range(1, pages + 1):
        listing_url = base_template.format(index=page)
        print(f"  Scraping page {page}/{pages} for {location_name}...")

        property_links = scrape_property_links(listing_url)
        print(f"  Found {len(property_links)} properties on page {page}")

        for link_idx, prop_url in enumerate(property_links, 1):
            property_data = scrape_property_details(prop_url)
            if property_data:
                data.append(property_data)

            if link_idx % 10 == 0:
                print(f"    Scraped {link_idx}/{len(property_links)} properties from page {page}")
                time.sleep(1)

        time.sleep(0.5)

# -------------------------
# Process single page URLs
# -------------------------
print(f"\nStarting to scrape {len(single_page_urls)} single-page URLs...")
for url_idx, listing_url in enumerate(single_page_urls, 1):
    location_name = listing_url.split('/')[-1].split('-')[0].replace('_', ' ')
    print(f"[{url_idx}/{len(single_page_urls)}] Processing {location_name}...")

    property_links = scrape_property_links(listing_url)
    print(f"  Found {len(property_links)} properties")

    for link_idx, prop_url in enumerate(property_links, 1):
        property_data = scrape_property_details(prop_url)
        if property_data:
            data.append(property_data)

        if link_idx % 10 == 0:
            print(f"    Scraped {link_idx}/{len(property_links)} properties")
            time.sleep(1)

    time.sleep(0.5)

# -------------------------
# Create final DataFrame
# -------------------------
print(f"\nScraping completed! Total properties collected: {len(data)}")
df = pd.DataFrame(data)

# ✅ Ensure ALL dynamic main feature columns exist
for k in all_feature_keys:
    if k not in df.columns:
        df[k] = None

print("\nDataFrame Info:")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

print("\nColumn names:")
for col in df.columns:
    print(f"  - {col}")

print("\nFirst 5 rows:")
print(df.head())

csv_filename = "lahore_flats.csv"
# df.to_csv(csv_filename, index=False)
print(f"\nData saved to {csv_filename}")

if not df.empty:
    print("\nBasic Statistics:")
    print(f"Price range: {df['Price'].nunique()} unique prices")
    print(f"Bedroom types: {df['Bedrooms'].nunique()} different bedroom counts")
    print(f"Areas covered: {df['Address'].nunique()} unique addresses")
    print(f"Average Marla: {df['Marla'].mean():.2f}" if df['Marla'].notna().any() else "No Marla data")
    print(f"Average Area (sqft): {df['Area (sqft)'].mean():.2f}" if df['Area (sqft)'].notna().any() else "No Area data")

print("\nScraping process completed successfully!")


Starting to scrape multi-page URLs...
[1/14] Processing Lahore Bahria Town - 32 pages...
  Scraping page 1/32 for Lahore Bahria Town...
  Found 0 properties on page 1
  Scraping page 2/32 for Lahore Bahria Town...
  Found 0 properties on page 2
  Scraping page 3/32 for Lahore Bahria Town...
  Found 0 properties on page 3
  Scraping page 4/32 for Lahore Bahria Town...
  Found 0 properties on page 4


KeyboardInterrupt: 

In [9]:
df.to_csv(csv_filename, index=False)

In [ ]:
response=requests.get("https://www.zameen.com/Property/askari_askari_10_brand_new_13_5_marla_4_bedroom_luxurious_flat_for_sale-53615259-1011-1.html")
soup=BeautifulSoup(response.content,"html.parser")

In [11]:
soup=BeautifulSoup(response.content,"html.parser")

In [42]:
list=soup.find("ul",class_="_49fc0232")

In [45]:
all_details=list.find_all(class_='_51519f00')
all_details

[<li class="_51519f00"><div class="_359e9f89"><div class="d0142259">Main Features</div></div><ul class="_3efd3392"><li class="_59261156"><svg class="_0dd20c8b"><use xlink:href="/assets/iconAmenities_noinline.8dbde8780eff18a63210a01d691e1c6b.svg#built-in-year"></use></svg><span class="_9121cbf9">Built in year<!-- -->: 2026</span></li><li class="_59261156"><svg class="_0dd20c8b"><use xlink:href="/assets/iconAmenities_noinline.8dbde8780eff18a63210a01d691e1c6b.svg#parking-spaces"></use></svg><span class="_9121cbf9">Parking Spaces<!-- -->: 2</span></li><li class="_59261156"><svg class="_0dd20c8b"><use xlink:href="/assets/iconAmenities_noinline.8dbde8780eff18a63210a01d691e1c6b.svg#lobby"></use></svg><span class="_9121cbf9">Lobby in Building</span></li><li class="_59261156"><svg class="_0dd20c8b"><use xlink:href="/assets/iconAmenities_noinline.8dbde8780eff18a63210a01d691e1c6b.svg#double-glazed-windows"></use></svg><span class="_9121cbf9">Double Glazed Windows</span></li><li class="_59261156">

In [54]:
features = []
for detail in all_details:
    all_list=detail.find_all('li',class_='_59261156')
    print(all_list)
    for li in all_list:
        span = li.find("span", class_="_9121cbf9")
        if not span:
            continue

        text = span.get_text(" ", strip=True)
        features.append(text)
    break
print(features)


[<li class="_59261156"><svg class="_0dd20c8b"><use xlink:href="/assets/iconAmenities_noinline.8dbde8780eff18a63210a01d691e1c6b.svg#built-in-year"></use></svg><span class="_9121cbf9">Built in year<!-- -->: 2026</span></li>, <li class="_59261156"><svg class="_0dd20c8b"><use xlink:href="/assets/iconAmenities_noinline.8dbde8780eff18a63210a01d691e1c6b.svg#parking-spaces"></use></svg><span class="_9121cbf9">Parking Spaces<!-- -->: 2</span></li>, <li class="_59261156"><svg class="_0dd20c8b"><use xlink:href="/assets/iconAmenities_noinline.8dbde8780eff18a63210a01d691e1c6b.svg#lobby"></use></svg><span class="_9121cbf9">Lobby in Building</span></li>, <li class="_59261156"><svg class="_0dd20c8b"><use xlink:href="/assets/iconAmenities_noinline.8dbde8780eff18a63210a01d691e1c6b.svg#double-glazed-windows"></use></svg><span class="_9121cbf9">Double Glazed Windows</span></li>, <li class="_59261156"><svg class="_0dd20c8b"><use xlink:href="/assets/iconAmenities_noinline.8dbde8780eff18a63210a01d691e1c6b.sv

In [94]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://www.zameen.com"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.zameen.com/",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Connection": "keep-alive"
})

# -----------------------------------------
# ✅ YOUR WORKING URL CONFIG (PAGES + URLs)
# -----------------------------------------
url_configs = [
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Town-509-{index}.html", "pages": 21},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Askari-3555-{index}.html", "pages": 37},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Gulberg-7-{index}.html", "pages": 22},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Raiwind_Road-128-{index}.html", "pages": 12},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_DHA_Defence-9-{index}.html", "pages": 10},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Orchard-1474-{index}.html", "pages": 4},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Main_Canal_Bank_Road-431-{index}.html", "pages": 5},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Johar_Town-93-{index}.html", "pages": 3},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Land_Breeze_Housing_Society-9784-{index}.html", "pages": 1},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Lakshmi_Chowk-1505-{index}.html", "pages": 2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Shah_Jamal-1491-{index}.html", "pages": 2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Defence_Road-4574-{index}.html", "pages": 2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Shanghai_Road-12334-{index}.html", "pages": 2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Ferozepur_Road-81-{index}.html", "pages": 1},
]

single_page_urls = [
    "https://www.zameen.com/Flats_Apartments/Lahore_Khayaban_e_Amin-1514-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Eden-3098-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Wapda_Town-423-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Garden_Town-4-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Jubilee_Town-766-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Khayaban_e_Jinnah_Road-8145-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Cooper_Road-12238-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Allama_Iqbal_Town-58-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Central_Park_Housing_Scheme-1013-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Sukh_Chayn_Gardens-473-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Nasheman-4510-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shadman-15803-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Jail_Road-528-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Ichhra-88-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Lahore_Villas-12049-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_New_Lahore_City-8152-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Park_Avenue_Housing_Scheme-10934-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Izmir_Town-483-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Model_Town-8-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Canal_Garden-1287-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Khayaban_e_Zafar-18840-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Multan_Road-48-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Kamahan_Road-8855-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Ring_Road-4575-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Lawrence_Road-8288-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_GT_Road-472-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Qartaba_Chowk-12099-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_State_Life_Housing_Society-487-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Kacha_Ferozepur_Road-11330-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Rehman_Gardens-4584-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Labor_Colony-10932-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Paragon_City-1014-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_IEP_Engineers_Town-8425-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Fazaia_Housing_Scheme-1677-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Thokar_Niaz_Baig-145-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Valencia_Housing_Society-373-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Maulana_Shaukat_Ali_Road-9866-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Icon_Valley-17703-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_LDA_Avenue-547-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Noor_Jahan_Road-10669-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Davis_Road-387-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shama_Road-12148-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Sabzazar_Scheme-562-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Ravi_Gardens_Smart_Homes-21814-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Lytton_Road-4581-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Punjab_Co_operative_Housing_Society-456-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Education___Medical_City-8334-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Cantt-427-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shabbir_Town-4389-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Islam_Pura-1369-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Samanabad-377-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Chauburji-13907-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Abdul_Sattar_Edhi_Road-10858-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Mustafa_Town-325-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Medical_Housing_Society-4359-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Faisal_Town-79-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shahkam_Chowk-11216-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_DHA_11_Rahbar-1410-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Revenue_Society-1359-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Kotha_Pind-10955-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Wafaqi_Colony-4380-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Walton_Road-153-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Punjab_Govt_Employees_Society-4430-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_LDA_Road-13991-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Pine_Avenue-14395-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_PIA_Housing_Scheme-444-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Iqbal_Avenue-1269-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Rehmanpura__Ferozpur_Road_-1683-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Mozang-8117-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Zaman_Colony-4472-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Band_Road-1212-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Pico_Road-21483-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shalimar_Link_Road-9437-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Muslim_Town-326-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Sant_Nagar-4537-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Super_Town-8504-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Saadi_Park-4483-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Waheed_Brother_Colony-4494-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_New_Garden_Town-16690-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_EME_Society-410-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Vital_Homes_Housing_Scheme-9461-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Cavalry_Ground-69-1.html"
]

# -----------------------------------------
# Storage
# -----------------------------------------
data = []
all_feature_keys = set()
seen_property_ids = set()

# -------------------------
# Label helpers
# -------------------------
def is_area_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) == "Area"

def is_bedroom_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) in ["Bedroom(s)", "Beds"]

def is_bath_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) in ["Bath(s)", "Baths"]

def get_soup(url, timeout=25):
    r = session.get(url, timeout=timeout)
    r.raise_for_status()
    return BeautifulSoup(r.content, "html.parser")

def clean(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

# -------------------------
# Listing links (robust)
# -------------------------
def scrape_property_links(listing_url: str):
    
    
    soup = get_soup(listing_url)
    ul = soup.find("ul", class_="e20beb46")
    links = []
    if ul:
        for a in ul.find_all("a", class_="d870ae17", href=True):
            links.append(urljoin(BASE, a["href"]))
    return links

# -------------------------
# Parse ALL sections
# -------------------------
def parse_all_sections(soup: BeautifulSoup):
    out = {}
    ul = soup.find("ul", class_="_49fc0232")
    if not ul:
        return out

    for sec in ul.find_all("li", class_="_51519f00"):
        header = sec.find("div", class_="_359e9f89")
        title = clean(header.get_text(" ", strip=True)) if header else None
        if not title:
            continue

        items = []
        items_ul = sec.find("ul", class_="_3efd3392")
        if items_ul:
            for li in items_ul.find_all("li", class_="_59261156"):
                span = li.find("span", class_="_9121cbf9")
                if span:
                    txt = clean(span.get_text(" ", strip=True))
                    if txt:
                        items.append(txt)

        if items:
            out[title] = items

    return out

def parse_rooms(section_map: dict):
    rooms_items = section_map.get("Rooms") or []
    room_details = {}
    other_rooms = []
    for it in rooms_items:
        if ":" in it:
            k, v = it.split(":", 1)
            room_details[clean(k)] = clean(v)
        else:
            other_rooms.append(it)
    return room_details, other_rooms

# -------------------------
# Property details (UPDATED)
# -------------------------
def scrape_property_details(url):
    global all_feature_keys

    try:
        prop_res = session.get(url, timeout=25)
        prop_res.raise_for_status()
        soup = BeautifulSoup(prop_res.content, "html.parser")

        name_tag = soup.find(class_='cd230541')
        price_tag = soup.find(class_='_2923a568')
        area_li = soup.find(is_area_li)
        bed_li = soup.find(is_bedroom_li)
        bath_li = soup.find(is_bath_li)

        bed_count = bath_count = marla_value = area_sqft = None
        name = price = ""

        # Beds/Baths
        if bed_li:
            bed_span = bed_li.find('span', class_='_2fdf7fc5')
            bed_count = clean(bed_span.get_text(strip=True)) if bed_span else None

        if bath_li:
            bath_span = bath_li.find('span', class_='_2fdf7fc5')
            bath_count = clean(bath_span.get_text(strip=True)) if bath_span else None

        # Name
        page_title = clean(name_tag.get_text(strip=True)) if name_tag else "Not listed"
        if bed_count:
            name = f"{bed_count} BHK {page_title}"
        else:
            name = page_title

        # Price
        price_raw = clean(price_tag.get_text(strip=True)) if price_tag else ""
        if price_raw.startswith("PKR") and not price_raw.startswith("PKR "):
            price = "PKR " + price_raw[3:].strip()
        else:
            price = price_raw

        # Area (Marla -> sqft)
        if area_li:
            area_value_span = area_li.find('span', class_='_2fdf7fc5')
            if area_value_span:
                area_text = clean(area_value_span.get_text(strip=True))
                m = re.search(r"([\d.]+)", area_text)
                if m:
                    marla_value = float(m.group(1))
                    area_sqft = marla_value * 225

        # Address
        address = page_title

        # ✅ Description
        desc_tag = soup.find(class_="_3547dac9")
        description = clean(desc_tag.get_text(" ", strip=True)) if desc_tag else None

        # ✅ ALL sections
        section_map = parse_all_sections(soup)

        # ✅ Nearby Locations column
        nearby_key = "Nearby Locations and Other Facilities"
        nearby_locations = ", ".join(section_map.get(nearby_key, [])) if section_map.get(nearby_key) else None

        # ✅ Rooms (same logic)
        room_details, other_rooms = parse_rooms(section_map)

        # ✅ Merge features from ALL sections except Rooms
        merged_features = []
        for sec_title, items in section_map.items():
            if sec_title == "Rooms":
                continue
            merged_features.extend(items)

        features_str = " | ".join(merged_features) if merged_features else None

        # ✅ Dynamic features dict (like your old main_features_dict, but correct)
        main_features_dict = {}
        for t in merged_features:
            if ":" in t:
                k, v = t.split(":", 1)
                k, v = clean(k), clean(v)
                if k:
                    main_features_dict[k] = v
            else:
                main_features_dict[clean(t)] = True

        all_feature_keys.update(main_features_dict.keys())

        # fixed columns from features
        floor_number = main_features_dict.get("Floor") or main_features_dict.get("Floor Number")
        total_floors = main_features_dict.get("Floors in Building")
        built_year = main_features_dict.get("Built in year")

        # Property ID
        match = re.search(r'-(\d+)-\d+-\d+\.html$', url)
        property_id = match.group(1) if match else "Not found"

        # Row
        row = {
            "Property ID": property_id,
            "Name": name,
            "Price": price,
            "Marla": marla_value,
            "Area (sqft)": area_sqft,
            "Bedrooms": bed_count,
            "Baths": bath_count,
            "Floor Number": floor_number,
            "Total Floors": total_floors,
            "Built Year": built_year,
            "Address": address,

            # ✅ NEW columns you asked
            "Description": description,
            "Features": features_str,
            "Nearby Locations and Other Facilities": nearby_locations,

            # ✅ Rooms same logic
            "Rooms": room_details,
            "Other Rooms": ", ".join(other_rooms),

            "Link": url
        }

        # ✅ Add every feature as separate column (dynamic)
        for k, v in main_features_dict.items():
            row[k] = v

        # OPTIONAL: each section separately (useful for debugging)
        for sec_title, items in section_map.items():
            row[f"Section - {sec_title}"] = ", ".join(items)

        return row

    except Exception as e:
        print(f"Error scraping property {url}: {e}")
        return None

# -------------------------
# RUN
# -------------------------
print("Starting to scrape multi-page URLs...")
total_configs = len(url_configs)

for config_idx, config in enumerate(url_configs, 1):
    base_template = config["base_url"]
    pages = config["pages"]
    location_name = base_template.split('/')[-1].split('-')[0].replace('_', ' ')

    print(f"\n[{config_idx}/{total_configs}] Processing {location_name} - {pages} pages...")

    for page in range(1, pages + 1):
        listing_url = base_template.format(index=page)
        print(f"  Scraping page {page}/{pages}: {listing_url}")

        property_links = scrape_property_links(listing_url)
        print(f"  Found {len(property_links)} properties on page {page}")

        for link_idx, prop_url in enumerate(property_links, 1):
            # de-dupe by Property ID
            m = re.search(r'-(\d+)-\d+-\d+\.html$', prop_url)
            pid = m.group(1) if m else None
            if pid and pid in seen_property_ids:
                continue

            property_data = scrape_property_details(prop_url)
            if property_data:
                data.append(property_data)
                if pid:
                    seen_property_ids.add(pid)

            if link_idx % 10 == 0:
                print(f"    Scraped {link_idx}/{len(property_links)} properties from page {page}")
                # time.sleep(0.5)

        # time.sleep(0.3)

print(f"\nStarting to scrape {len(single_page_urls)} single-page URLs...")
for url_idx, listing_url in enumerate(single_page_urls, 1):
    location_name = listing_url.split('/')[-1].split('-')[0].replace('_', ' ')
    print(f"\n[{url_idx}/{len(single_page_urls)}] Processing {location_name}: {listing_url}")

    property_links = scrape_property_links(listing_url)
    print(f"  Found {len(property_links)} properties")

    for link_idx, prop_url in enumerate(property_links, 1):
        m = re.search(r'-(\d+)-\d+-\d+\.html$', prop_url)
        pid = m.group(1) if m else None
        if pid and pid in seen_property_ids:
            continue

        property_data = scrape_property_details(prop_url)
        if property_data:
            data.append(property_data)
            if pid:
                seen_property_ids.add(pid)

        if link_idx % 10 == 0:
            print(f"    Scraped {link_idx}/{len(property_links)} properties")
            # time.sleep(0.5)

    # time.sleep(0.3)

# -------------------------
# Create final DataFrame
# -------------------------
print(f"\nScraping completed! Total properties collected: {len(data)}")
df = pd.DataFrame(data)

# ensure ALL dynamic feature columns exist
for k in all_feature_keys:
    if k not in df.columns:
        df[k] = None

print("\nDataFrame Info:")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

print("\nFirst 5 rows:")
print(df.head())

# df.to_csv("lahore_flats.csv", index=False, encoding="utf-8-sig")
print("\n✅ Saved columns include: Description, Features, Nearby Locations and Other Facilities")


Starting to scrape multi-page URLs...

[1/14] Processing Lahore Bahria Town - 21 pages...
  Scraping page 1/21: https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Town-509-1.html
  Found 27 properties on page 1
    Scraped 10/27 properties from page 1
    Scraped 20/27 properties from page 1
  Scraping page 2/21: https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Town-509-2.html
  Found 27 properties on page 2
    Scraped 10/27 properties from page 2
    Scraped 20/27 properties from page 2
  Scraping page 3/21: https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Town-509-3.html
  Found 27 properties on page 3
    Scraped 10/27 properties from page 3
    Scraped 20/27 properties from page 3
  Scraping page 4/21: https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Town-509-4.html
  Found 27 properties on page 4
    Scraped 10/27 properties from page 4
    Scraped 20/27 properties from page 4
  Scraping page 5/21: https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Town-509-5

HTTPError: 404 Client Error: Not Found for url: https://www.zameen.com/Flats_Apartments/Lahore_Lakshmi_Chowk-1505-1.html

In [90]:
df.columns

Index(['Property ID', 'Name', 'Price', 'Marla', 'Area (sqft)', 'Bedrooms',
       'Baths', 'Floor Number', 'Total Floors', 'Built Year', 'Parking Spaces',
       'Address', 'Rooms', 'Other Rooms', 'Link', 'Built in year', 'Floor',
       'Floors in Building', 'Elevators', 'Bathrooms', 'Servant Quarters',
       'Kitchens', 'Store Rooms'],
      dtype='object')

In [ ]:
response=requests.get("https://www.zameen.com/Property/askari_askari_10_brand_new_13_5_marla_4_bedroom_luxurious_flat_for_sale-53615259-1011-1.html")
soup=BeautifulSoup(response.content,"html.parser")

In [1]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://www.zameen.com"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.zameen.com/",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Connection": "keep-alive"
})

# -----------------------------------------
# ✅ YOUR WORKING URL CONFIG (PAGES + URLs)
# -----------------------------------------
url_configs = [
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Town-509-{index}.html", "pages": 1},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Askari-3555-{index}.html", "pages": 37},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Gulberg-7-{index}.html", "pages": 22},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Raiwind_Road-128-{index}.html", "pages": 12},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_DHA_Defence-9-{index}.html", "pages": 10},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Orchard-1474-{index}.html", "pages": 4},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Main_Canal_Bank_Road-431-{index}.html", "pages": 5},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Johar_Town-93-{index}.html", "pages": 3},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Land_Breeze_Housing_Society-9784-{index}.html", "pages": 1},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Lakshmi_Chowk-1505-{index}.html", "pages": 2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Shah_Jamal-1491-{index}.html", "pages": 2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Defence_Road-4574-{index}.html", "pages": 2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Shanghai_Road-12334-{index}.html", "pages": 2},
    {"base_url": "https://www.zameen.com/Flats_Apartments/Lahore_Ferozepur_Road-81-{index}.html", "pages": 1},
]

single_page_urls = [
    "https://www.zameen.com/Flats_Apartments/Lahore_Khayaban_e_Amin-1514-1.html"
    "https://www.zameen.com/Flats_Apartments/Lahore_Eden-3098-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Wapda_Town-423-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Garden_Town-4-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Jubilee_Town-766-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Khayaban_e_Jinnah_Road-8145-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Cooper_Road-12238-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Allama_Iqbal_Town-58-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Central_Park_Housing_Scheme-1013-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Sukh_Chayn_Gardens-473-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Nasheman-4510-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shadman-15803-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Jail_Road-528-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Ichhra-88-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Lahore_Villas-12049-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_New_Lahore_City-8152-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Park_Avenue_Housing_Scheme-10934-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Izmir_Town-483-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Model_Town-8-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Canal_Garden-1287-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Khayaban_e_Zafar-18840-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Multan_Road-48-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Kamahan_Road-8855-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Ring_Road-4575-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Lawrence_Road-8288-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_GT_Road-472-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Qartaba_Chowk-12099-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_State_Life_Housing_Society-487-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Kacha_Ferozepur_Road-11330-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Rehman_Gardens-4584-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Labor_Colony-10932-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Paragon_City-1014-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_IEP_Engineers_Town-8425-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Fazaia_Housing_Scheme-1677-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Thokar_Niaz_Baig-145-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Valencia_Housing_Society-373-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Maulana_Shaukat_Ali_Road-9866-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Icon_Valley-17703-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_LDA_Avenue-547-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Noor_Jahan_Road-10669-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Davis_Road-387-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shama_Road-12148-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Sabzazar_Scheme-562-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Ravi_Gardens_Smart_Homes-21814-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Lytton_Road-4581-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Punjab_Co_operative_Housing_Society-456-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Education___Medical_City-8334-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Cantt-427-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shabbir_Town-4389-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Islam_Pura-1369-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Samanabad-377-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Chauburji-13907-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Abdul_Sattar_Edhi_Road-10858-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Mustafa_Town-325-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Medical_Housing_Society-4359-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Faisal_Town-79-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shahkam_Chowk-11216-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_DHA_11_Rahbar-1410-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Revenue_Society-1359-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Kotha_Pind-10955-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Wafaqi_Colony-4380-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Walton_Road-153-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Punjab_Govt_Employees_Society-4430-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_LDA_Road-13991-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Pine_Avenue-14395-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_PIA_Housing_Scheme-444-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Iqbal_Avenue-1269-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Rehmanpura__Ferozpur_Road_-1683-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Mozang-8117-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Zaman_Colony-4472-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Band_Road-1212-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Pico_Road-21483-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Shalimar_Link_Road-9437-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Muslim_Town-326-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Sant_Nagar-4537-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Super_Town-8504-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Saadi_Park-4483-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Waheed_Brother_Colony-4494-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_New_Garden_Town-16690-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_EME_Society-410-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Vital_Homes_Housing_Scheme-9461-1.html",
    "https://www.zameen.com/Flats_Apartments/Lahore_Cavalry_Ground-69-1.html"
]

# -----------------------------------------
# Storage
# -----------------------------------------
data = []
all_feature_keys = set()
seen_property_ids = set()

# Error tracking
failed_pages = []
failed_properties = []

# -------------------------
# Label helpers
# -------------------------
def is_area_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) == "Area"

def is_bedroom_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) in ["Bedroom(s)", "Beds"]

def is_bath_li(tag):
    if tag.name != 'li':
        return False
    label_span = tag.find('span', class_='ed0db22a')
    return label_span and label_span.get_text(strip=True) in ["Bath(s)", "Baths"]

def get_soup(url, timeout=25):
    """Get soup with error handling"""
    try:
        r = session.get(url, timeout=timeout)
        r.raise_for_status()
        return BeautifulSoup(r.content, "html.parser")
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            print(f"  ⚠️  Page not found (404): {url}")
        else:
            print(f"  ⚠️  HTTP Error {e.response.status_code}: {url}")
        return None
    except requests.exceptions.Timeout:
        print(f"  ⚠️  Timeout error: {url}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"  ⚠️  Request error: {url} - {str(e)}")
        return None
    except Exception as e:
        print(f"  ⚠️  Unexpected error: {url} - {str(e)}")
        return None

def clean(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

# -------------------------
# Listing links (robust)
# -------------------------
def scrape_property_links(listing_url: str):
    """Scrape property links with error handling"""
    try:
        soup = get_soup(listing_url)
        if soup is None:
            return []
        
        ul = soup.find("ul", class_="e20beb46")
        links = []
        if ul:
            for a in ul.find_all("a", class_="d870ae17", href=True):
                links.append(urljoin(BASE, a["href"]))
        return links
    except Exception as e:
        print(f"  ⚠️  Error extracting property links from {listing_url}: {str(e)}")
        return []

# -------------------------
# Parse ALL sections
# -------------------------
def parse_all_sections(soup: BeautifulSoup):
    out = {}
    ul = soup.find("ul", class_="_49fc0232")
    if not ul:
        return out

    for sec in ul.find_all("li", class_="_51519f00"):
        header = sec.find("div", class_="_359e9f89")
        title = clean(header.get_text(" ", strip=True)) if header else None
        if not title:
            continue

        items = []
        items_ul = sec.find("ul", class_="_3efd3392")
        if items_ul:
            for li in items_ul.find_all("li", class_="_59261156"):
                span = li.find("span", class_="_9121cbf9")
                if span:
                    txt = clean(span.get_text(" ", strip=True))
                    if txt:
                        items.append(txt)

        if items:
            out[title] = items

    return out

def parse_rooms(section_map: dict):
    rooms_items = section_map.get("Rooms") or []
    room_details = {}
    other_rooms = []
    for it in rooms_items:
        if ":" in it:
            k, v = it.split(":", 1)
            room_details[clean(k)] = clean(v)
        else:
            other_rooms.append(it)
    return room_details, other_rooms

# -------------------------
# Property details (with error handling)
# -------------------------
# Counter for first fetch
first_fetch_printed = False

def scrape_property_details(url):
    global all_feature_keys, first_fetch_printed

    try:
        soup = get_soup(url)
        if soup is None:
            failed_properties.append({"url": url, "error": "Failed to fetch page"})
            return None

        name_tag = soup.find(class_='cd230541')
        price_tag = soup.find(class_='_2923a568')
        area_li = soup.find(is_area_li)
        bed_li = soup.find(is_bedroom_li)
        bath_li = soup.find(is_bath_li)

        bed_count = bath_count = marla_value = area_sqft = None
        name = price = ""

        # Beds/Baths
        if bed_li:
            bed_span = bed_li.find('span', class_='_2fdf7fc5')
            bed_count = clean(bed_span.get_text(strip=True)) if bed_span else None

        if bath_li:
            bath_span = bath_li.find('span', class_='_2fdf7fc5')
            bath_count = clean(bath_span.get_text(strip=True)) if bath_span else None

        # Name
        page_title = clean(name_tag.get_text(strip=True)) if name_tag else "Not listed"
        if bed_count:
            name = f"{bed_count} BHK {page_title}"
        else:
            name = page_title

        # Price
        price_raw = clean(price_tag.get_text(strip=True)) if price_tag else ""
        if price_raw.startswith("PKR") and not price_raw.startswith("PKR "):
            price = "PKR " + price_raw[3:].strip()
        else:
            price = price_raw

        # Area (Marla -> sqft)
        if area_li:
            area_value_span = area_li.find('span', class_='_2fdf7fc5')
            if area_value_span:
                area_text = clean(area_value_span.get_text(strip=True))
                m = re.search(r"([\d.]+)", area_text)
                if m:
                    marla_value = float(m.group(1))
                    area_sqft = marla_value * 225

        # Address
        address = page_title

        # Description
        desc_tag = soup.find(class_="_3547dac9")
        description = clean(desc_tag.get_text(" ", strip=True)) if desc_tag else None

        # ALL sections
        section_map = parse_all_sections(soup)

        # Nearby Locations column
        nearby_key = "Nearby Locations and Other Facilities"
        nearby_locations = ", ".join(section_map.get(nearby_key, [])) if section_map.get(nearby_key) else None

        # Rooms
        room_details, other_rooms = parse_rooms(section_map)

        # Merge features from ALL sections except Rooms
        merged_features = []
        for sec_title, items in section_map.items():
            if sec_title == "Rooms":
                continue
            merged_features.extend(items)

        features_str = " | ".join(merged_features) if merged_features else None

        # Dynamic features dict
        main_features_dict = {}
        for t in merged_features:
            if ":" in t:
                k, v = t.split(":", 1)
                k, v = clean(k), clean(v)
                if k:
                    main_features_dict[k] = v
            else:
                main_features_dict[clean(t)] = True

        all_feature_keys.update(main_features_dict.keys())

        # fixed columns from features
        floor_number = main_features_dict.get("Floor") or main_features_dict.get("Floor Number")
        total_floors = main_features_dict.get("Floors in Building")
        built_year = main_features_dict.get("Built in year")

        # Property ID
        match = re.search(r'-(\d+)-\d+-\d+\.html$', url)
        property_id = match.group(1) if match else "Not found"

        # Row
        row = {
            "Property ID": property_id,
            "Name": name,
            "Price": price,
            "Marla": marla_value,
            "Area (sqft)": area_sqft,
            "Bedrooms": bed_count,
            "Baths": bath_count,
            "Floor Number": floor_number,
            "Total Floors": total_floors,
            "Built Year": built_year,
            "Address": address,
            "Description": description,
            "Features": features_str,
            "Nearby Locations and Other Facilities": nearby_locations,
            "Rooms": room_details,
            "Other Rooms": ", ".join(other_rooms),
            "Link": url
        }

        # Add every feature as separate column
        for k, v in main_features_dict.items():
            row[k] = v

        # Each section separately
        for sec_title, items in section_map.items():
            row[f"Section - {sec_title}"] = ", ".join(items)

        # ✅ PRINT FIRST FETCH DETAILS
        if not first_fetch_printed:
            print("\n" + "=" * 80)
            print("🔍 FIRST PROPERTY FETCH - DETAILED OUTPUT")
            print("=" * 80)
            print(f"\n📍 URL: {url}")
            print(f"\n📝 BASIC INFO:")
            print(f"   Property ID: {property_id}")
            print(f"   Name: {name}")
            print(f"   Price: {price}")
            print(f"   Bedrooms: {bed_count}")
            print(f"   Baths: {bath_count}")
            print(f"   Marla: {marla_value}")
            print(f"   Area (sqft): {area_sqft}")
            print(f"   Address: {address}")
            
            print(f"\n📄 DESCRIPTION:")
            print(f"   {description[:200] if description else 'None'}..." if description and len(description) > 200 else f"   {description}")
            
            print(f"\n🏗️ BUILDING INFO:")
            print(f"   Floor Number: {floor_number}")
            print(f"   Total Floors: {total_floors}")
            print(f"   Built Year: {built_year}")
            
            print(f"\n🛋️ ROOMS:")
            print(f"   Room Details: {room_details}")
            print(f"   Other Rooms: {other_rooms}")
            
            print(f"\n✨ ALL SECTIONS FOUND:")
            for sec_title, items in section_map.items():
                print(f"   [{sec_title}]:")
                for item in items[:5]:  # Show first 5 items
                    print(f"      - {item}")
                if len(items) > 5:
                    print(f"      ... and {len(items) - 5} more items")
            
            print(f"\n🎯 FEATURES STRING (first 300 chars):")
            print(f"   {features_str[:300] if features_str else 'None'}...")
            
            print(f"\n📍 NEARBY LOCATIONS:")
            print(f"   {nearby_locations[:200] if nearby_locations else 'None'}..." if nearby_locations and len(nearby_locations) > 200 else f"   {nearby_locations}")
            
            print(f"\n🔑 DYNAMIC FEATURE KEYS EXTRACTED:")
            for k, v in list(main_features_dict.items())[:10]:  # Show first 10
                print(f"   {k}: {v}")
            if len(main_features_dict) > 10:
                print(f"   ... and {len(main_features_dict) - 10} more features")
            
            print(f"\n📊 TOTAL COLUMNS IN ROW: {len(row)}")
            print("=" * 80 + "\n")
            
            first_fetch_printed = True

        return row

    except Exception as e:
        error_msg = f"{type(e).__name__}: {str(e)}"
        print(f"  ⚠️  Error scraping property {url}: {error_msg}")
        failed_properties.append({"url": url, "error": error_msg})
        return None

# -------------------------
# RUN
# -------------------------
print("Starting to scrape multi-page URLs...")
print("=" * 80)
total_configs = len(url_configs)

for config_idx, config in enumerate(url_configs, 1):
    base_template = config["base_url"]
    pages = config["pages"]
    location_name = base_template.split('/')[-1].split('-')[0].replace('_', ' ')

    print(f"\n[{config_idx}/{total_configs}] Processing {location_name} - {pages} pages...")

    for page in range(1, pages + 1):
        listing_url = base_template.format(index=page)
        print(f"  Scraping page {page}/{pages}: {listing_url}")

        property_links = scrape_property_links(listing_url)
        
        if not property_links:
            print(f"  ⚠️  No properties found or page unavailable - skipping...")
            failed_pages.append({"url": listing_url, "reason": "No properties found"})
            continue
            
        print(f"  Found {len(property_links)} properties on page {page}")

        for link_idx, prop_url in enumerate(property_links, 1):
            # de-dupe by Property ID
            m = re.search(r'-(\d+)-\d+-\d+\.html$', prop_url)
            pid = m.group(1) if m else None
            if pid and pid in seen_property_ids:
                continue

            property_data = scrape_property_details(prop_url)
            if property_data:
                data.append(property_data)
                if pid:
                    seen_property_ids.add(pid)

            if link_idx % 10 == 0:
                print(f"    Scraped {link_idx}/{len(property_links)} properties from page {page}")

print(f"\n{'=' * 80}")
print(f"Starting to scrape {len(single_page_urls)} single-page URLs...")
print("=" * 80)

for url_idx, listing_url in enumerate(single_page_urls, 1):
    location_name = listing_url.split('/')[-1].split('-')[0].replace('_', ' ')
    print(f"\n[{url_idx}/{len(single_page_urls)}] Processing {location_name}: {listing_url}")

    property_links = scrape_property_links(listing_url)
    
    if not property_links:
        print(f"  ⚠️  No properties found or page unavailable - skipping...")
        failed_pages.append({"url": listing_url, "reason": "No properties found"})
        continue
        
    print(f"  Found {len(property_links)} properties")

    for link_idx, prop_url in enumerate(property_links, 1):
        m = re.search(r'-(\d+)-\d+-\d+\.html$', prop_url)
        pid = m.group(1) if m else None
        if pid and pid in seen_property_ids:
            continue

        property_data = scrape_property_details(prop_url)
        if property_data:
            data.append(property_data)
            if pid:
                seen_property_ids.add(pid)

        if link_idx % 10 == 0:
            print(f"    Scraped {link_idx}/{len(property_links)} properties")

# -------------------------
# Create final DataFrame
# -------------------------
print(f"\n{'=' * 80}")
print("SCRAPING SUMMARY")
print("=" * 80)
print(f"✅ Total properties collected: {len(data)}")
print(f"❌ Failed pages: {len(failed_pages)}")
print(f"❌ Failed properties: {len(failed_properties)}")

if failed_pages:
    print(f"\n⚠️  Failed Pages ({len(failed_pages)}):")
    for fp in failed_pages[:10]:  # Show first 10
        print(f"   - {fp['url']}")
    if len(failed_pages) > 10:
        print(f"   ... and {len(failed_pages) - 10} more")

if failed_properties:
    print(f"\n⚠️  Failed Properties ({len(failed_properties)}):")
    for fp in failed_properties[:10]:  # Show first 10
        print(f"   - {fp['url']}: {fp['error']}")
    if len(failed_properties) > 10:
        print(f"   ... and {len(failed_properties) - 10} more")

df = pd.DataFrame(data)

# ensure ALL dynamic feature columns exist
for k in all_feature_keys:
    if k not in df.columns:
        df[k] = None

print(f"\n{'=' * 80}")
print("DATAFRAME INFO")
print("=" * 80)
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

print("\nFirst 5 rows:")
print(df.head())

# Save to CSV
output_file = "lahore_flats_v2.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")
# print(f"\n✅ Data saved to {output_file}")
print("✅ Columns include: Description, Features, Nearby Locations and Other Facilities")

# Save error logs if any
if failed_pages or failed_properties:
    error_log = {
        "failed_pages": failed_pages,
        "failed_properties": failed_properties
    }
    import json
    with open("scraping_errors.json", "w") as f:
        json.dump(error_log, f, indent=2)
    print("⚠️  Error log saved to scraping_errors.json")

Starting to scrape multi-page URLs...

[1/14] Processing Lahore Bahria Town - 1 pages...
  Scraping page 1/1: https://www.zameen.com/Flats_Apartments/Lahore_Bahria_Town-509-1.html
  Found 27 properties on page 1

🔍 FIRST PROPERTY FETCH - DETAILED OUTPUT

📍 URL: https://www.zameen.com/Property/bahria_town_sector_e_bahria_town_-_johar_block_own_your_slice_of_luxury_studio_apartments_in_bahria_town_grand_11_easy_installments-53460817-1809-1.html

📝 BASIC INFO:
   Property ID: 53460817
   Name: - BHK Bahria Town - Johar Block, Bahria Town - Sector E, Bahria Town, Lahore, Punjab
   Price: PKR 54 Lakh
   Bedrooms: -
   Baths: 1
   Marla: 1.3
   Area (sqft): 292.5
   Address: Bahria Town - Johar Block, Bahria Town - Sector E, Bahria Town, Lahore, Punjab

📄 DESCRIPTION:
   Experience luxury living at Grand 11, a prestigious high-rise development in Bahria Town, Pakistans most sought-after community, known for its world-class infrastructure, security, and modern lifestyl...

🏗️ BUILDING INFO:
 

In [2]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://www.zameen.com"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.zameen.com/",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Connection": "keep-alive"
})

# -------------------------------------------------------
# ✅ PASTE YOUR SINGLE PROPERTY URL HERE
# -------------------------------------------------------
TARGET_URL = "https://www.zameen.com/Property/bahria_town_sector_e_bahria_town_-_nishtar_block_1_bed_brand_new_luxury_fully_furnished_apartment_available_for_sale_in_bahria_town_lahore-53537710-1806-1.html"
def get_soup(url, timeout=25):
    try:
        r = session.get(url, timeout=timeout)
        r.raise_for_status()
        return BeautifulSoup(r.content, "html.parser")
    except requests.exceptions.HTTPError as e:
        print(f"  ⚠️  HTTP Error {e.response.status_code}: {url}")
        return None
    except Exception as e:
        print(f"  ⚠️  Error fetching {url}: {e}")
        return None


def clean(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()


# -------------------------------------------------------
# ✅ Extract Beds & Baths from the TOP SUMMARY STRIP
#    Targets: <span aria-label="Beds">  and  <span aria-label="Baths">
#    This is the circled section in your screenshot.
# -------------------------------------------------------
def extract_beds_baths_area(soup):
    beds = baths = area_text = None

    # The summary strip div: class _14f36d85
    strip = soup.find("div", class_="_14f36d85")
    if strip:
        beds_span = strip.find("span", {"aria-label": "Beds"})
        baths_span = strip.find("span", {"aria-label": "Baths"})
        area_span = strip.find("span", {"aria-label": "Area"})

        if beds_span:
            # e.g. "1 Bed" → extract number
            m = re.search(r"(\d+)", beds_span.get_text(strip=True))
            beds = m.group(1) if m else clean(beds_span.get_text(strip=True))

        if baths_span:
            m = re.search(r"(\d+)", baths_span.get_text(strip=True))
            baths = m.group(1) if m else clean(baths_span.get_text(strip=True))

        if area_span:
            area_text = clean(area_span.get_text(strip=True))

    return beds, baths, area_text


def parse_area(area_text):
    """Convert area text to marla value and sqft."""
    marla_value = area_sqft = None
    if area_text:
        m = re.search(r"([\d.]+)", area_text)
        if m:
            marla_value = float(m.group(1))
            area_sqft = marla_value * 225
    return marla_value, area_sqft


def parse_all_sections(soup):
    out = {}
    ul = soup.find("ul", class_="_49fc0232")
    if not ul:
        return out
    for sec in ul.find_all("li", class_="_51519f00"):
        header = sec.find("div", class_="_359e9f89")
        title = clean(header.get_text(" ", strip=True)) if header else None
        if not title:
            continue
        items = []
        items_ul = sec.find("ul", class_="_3efd3392")
        if items_ul:
            for li in items_ul.find_all("li", class_="_59261156"):
                span = li.find("span", class_="_9121cbf9")
                if span:
                    txt = clean(span.get_text(" ", strip=True))
                    if txt:
                        items.append(txt)
        if items:
            out[title] = items
    return out


def parse_rooms(section_map):
    rooms_items = section_map.get("Rooms") or []
    room_details = {}
    other_rooms = []
    for it in rooms_items:
        if ":" in it:
            k, v = it.split(":", 1)
            room_details[clean(k)] = clean(v)
        else:
            other_rooms.append(it)
    return room_details, other_rooms


def scrape_property(url):
    print(f"\n🔍 Fetching: {url}")
    soup = get_soup(url)
    if soup is None:
        print("❌ Could not fetch page.")
        return None

    # ✅ Beds / Baths / Area — from the TOP SUMMARY STRIP (circled in screenshot)
    beds, baths, area_text = extract_beds_baths_area(soup)
    marla_value, area_sqft = parse_area(area_text)

    # Title
    name_tag = soup.find(class_="cd230541")
    page_title = clean(name_tag.get_text(strip=True)) if name_tag else "Not listed"

    # Build name with bed count prefix
    name = f"{beds} BHK {page_title}" if beds else page_title

    # Price
    price_tag = soup.find(class_="_2923a568")
    price_raw = clean(price_tag.get_text(strip=True)) if price_tag else ""
    if price_raw.startswith("PKR") and not price_raw.startswith("PKR "):
        price = "PKR " + price_raw[3:].strip()
    else:
        price = price_raw

    # Description
    desc_tag = soup.find(class_="_3547dac9")
    description = clean(desc_tag.get_text(" ", strip=True)) if desc_tag else None

    # All sections
    section_map = parse_all_sections(soup)

    # Nearby Locations
    nearby_key = "Nearby Locations and Other Facilities"
    nearby_locations = ", ".join(section_map.get(nearby_key, [])) or None

    # Rooms
    room_details, other_rooms = parse_rooms(section_map)

    # Merge features from all sections except Rooms
    merged_features = []
    for sec_title, items in section_map.items():
        if sec_title == "Rooms":
            continue
        merged_features.extend(items)

    features_str = " | ".join(merged_features) if merged_features else None

    # Dynamic features dict
    main_features_dict = {}
    for t in merged_features:
        if ":" in t:
            k, v = t.split(":", 1)
            k, v = clean(k), clean(v)
            if k:
                main_features_dict[k] = v
        else:
            main_features_dict[clean(t)] = True

    floor_number = main_features_dict.get("Floor") or main_features_dict.get("Floor Number")
    total_floors = main_features_dict.get("Floors in Building")
    built_year = main_features_dict.get("Built in year")

    # Property ID from URL
    match = re.search(r'-(\d+)-\d+\.html$', url)
    property_id = match.group(1) if match else "Not found"

    row = {
        "Property ID":   property_id,
        "Name":          name,
        "Price":         price,
        "Marla":         marla_value,
        "Area (sqft)":   area_sqft,
        "Bedrooms":      beds,       # ✅ separate column — from aria-label="Beds"
        "Baths":         baths,      # ✅ separate column — from aria-label="Baths"
        "Floor Number":  floor_number,
        "Total Floors":  total_floors,
        "Built Year":    built_year,
        "Address":       page_title,
        "Description":   description,
        "Features":      features_str,
        "Nearby Locations and Other Facilities": nearby_locations,
        "Rooms":         str(room_details),
        "Other Rooms":   ", ".join(other_rooms),
        "Link":          url,
    }

    # Add every feature as a separate column
    for k, v in main_features_dict.items():
        row[k] = v

    # Each section separately
    for sec_title, items in section_map.items():
        row[f"Section - {sec_title}"] = ", ".join(items)

    return row


# -------------------------------------------------------
# RUN
# -------------------------------------------------------
result = scrape_property(TARGET_URL)

if result:
    print("\n" + "=" * 70)
    print("✅ SCRAPED PROPERTY DETAILS")
    print("=" * 70)
    for k, v in result.items():
        if v is not None and v != "":
            val_str = str(v)
            print(f"  {k:<45} {val_str[:80]}")
    print("=" * 70)

    df = pd.DataFrame([result])
    output_file = "zameen_property.csv"
    df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"\n✅ Saved to {output_file}")
    print(f"   Rows: {len(df)}  |  Columns: {len(df.columns)}")
else:
    print("\n❌ No data scraped.")


🔍 Fetching: https://www.zameen.com/Property/bahria_town_sector_e_bahria_town_-_nishtar_block_1_bed_brand_new_luxury_fully_furnished_apartment_available_for_sale_in_bahria_town_lahore-53537710-1806-1.html

✅ SCRAPED PROPERTY DETAILS
  Property ID                                   1806
  Name                                          1 BHK Bahria Town - Nishtar Block, Bahria Town - Sector E, Bahria Town, Lahore, 
  Price                                         PKR 70 Lakh
  Marla                                         3.1
  Area (sqft)                                   697.5
  Bedrooms                                      1
  Baths                                         1
  Floor Number                                  2
  Total Floors                                  9
  Built Year                                    2020
  Address                                       Bahria Town - Nishtar Block, Bahria Town - Sector E, Bahria Town, Lahore, Punjab
  Description                        

## 2nd approach

In [2]:
import re
import time
import random
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE = "https://www.zameen.com"

# -------------------------------------------------------
# Session with retry logic
# -------------------------------------------------------
session = requests.Session()
retry_strategy = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["HEAD", "GET", "OPTIONS"]
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.zameen.com/",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
})

# -------------------------------------------------------
# ✅ MASTER URL FOR FLATS
# -------------------------------------------------------
MASTER_URL = "https://www.zameen.com/Flats_Apartments/Lahore-1-1.html"

# -------------------------------------------------------
# State
# -------------------------------------------------------
data = []
all_feature_keys = set()
seen_property_ids = set()
failed_pages = []
failed_properties = []
first_fetch_printed = False
request_count = 0
last_request_time = time.time()


# -------------------------------------------------------
# Rate limiting
# -------------------------------------------------------
def rate_limit():
    global request_count, last_request_time
    current_time = time.time()
    time_since_last = current_time - last_request_time
    delay = random.uniform(1.0, 2.0)
    if time_since_last < delay:
        time.sleep(delay - time_since_last)
    last_request_time = time.time()
    request_count += 1
    if request_count % 50 == 0:
        print(f"\n   💤 Taking 10s break after {request_count} requests...\n")
        time.sleep(10)


# -------------------------------------------------------
# Fetch helpers
# -------------------------------------------------------
def clean(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()


def get_soup(url, timeout=30, max_retries=3):
    for attempt in range(max_retries):
        try:
            rate_limit()
            r = session.get(url, timeout=timeout)
            r.raise_for_status()
            return BeautifulSoup(r.content, "html.parser")
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 404:
                return None
            if attempt < max_retries - 1:
                wait = (attempt + 1) * 5
                print(f"  ⚠️  HTTP {e.response.status_code}, retry in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  ⚠️  HTTP {e.response.status_code} after {max_retries} attempts: {url}")
                return None
        except requests.exceptions.Timeout:
            if attempt < max_retries - 1:
                wait = (attempt + 1) * 5
                print(f"  ⚠️  Timeout, retry in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  ⚠️  Timeout after {max_retries} attempts: {url}")
                return None
        except (requests.exceptions.ConnectionError, requests.exceptions.ChunkedEncodingError):
            if attempt < max_retries - 1:
                wait = (attempt + 1) * 10
                print(f"  ⚠️  Connection error, retry in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  ⚠️  Connection failed after {max_retries} attempts: {url}")
                return None
        except Exception as e:
            print(f"  ⚠️  Unexpected error: {str(e)[:100]}")
            return None
    return None


def make_page_url(seed_url: str, page: int) -> str:
    if re.search(r"-\d+\.html$", seed_url):
        return re.sub(r"-\d+\.html$", f"-{page}.html", seed_url)
    return seed_url


# -------------------------------------------------------
# 1) Extract all societies from the master listing page
# -------------------------------------------------------
def extract_societies(master_url: str):
    print(f"📄 Fetching master page: {master_url}")
    soup = get_soup(master_url)
    if soup is None:
        print("❌ Failed to fetch master page.")
        return []

    wrapper = soup.find(class_="_5d76b098")
    if not wrapper:
        print("⚠️  Could not find societies wrapper (_5d76b098).")
        return []

    items = wrapper.find_all("li", class_="_6d61972c")
    if not items:
        print("⚠️  No society <li> items found.")
        return []

    societies = []
    for li in items:
        a = li.find("a", class_="_6de4d4f7", href=True)
        if not a:
            continue
        href = urljoin(BASE, a["href"])
        full_text = a.get_text(" ", strip=True)
        name = re.sub(r"\(\s*[\d,]+\s*\)\s*$", "", full_text).strip()
        m = re.search(r"\(\s*([\d,]+)\s*\)\s*$", full_text)
        count = int(m.group(1).replace(",", "")) if m else None
        societies.append({"society_name": name, "society_count": count, "society_url": href})

    return societies


# -------------------------------------------------------
# 2) Listing page -> property links
# -------------------------------------------------------
def scrape_property_links(listing_url: str):
    try:
        soup = get_soup(listing_url)
        if soup is None:
            return []
        links = []
        ul = soup.find("ul", class_="e20beb46")
        if ul:
            for a in ul.find_all("a", class_="d870ae17", href=True):
                links.append(urljoin(BASE, a["href"]))
        return list(dict.fromkeys(links))  # dedupe, preserve order
    except Exception as e:
        print(f"  ⚠️  Error extracting links: {str(e)[:100]}")
        return []


# -------------------------------------------------------
# ✅ BEDS / BATHS / AREA — from the top summary strip
#    Targets aria-label="Beds", aria-label="Baths", aria-label="Area"
#    This is the circled section: 1 Bed | 1 Bath | 3.1 Marla
# -------------------------------------------------------
def extract_beds_baths_area(soup):
    beds = baths = area_text = None
    strip = soup.find("div", class_="_14f36d85")
    if strip:
        beds_span  = strip.find("span", {"aria-label": "Beds"})
        baths_span = strip.find("span", {"aria-label": "Baths"})
        area_span  = strip.find("span", {"aria-label": "Area"})

        if beds_span:
            m = re.search(r"(\d+)", beds_span.get_text(strip=True))
            beds = m.group(1) if m else clean(beds_span.get_text(strip=True))

        if baths_span:
            m = re.search(r"(\d+)", baths_span.get_text(strip=True))
            baths = m.group(1) if m else clean(baths_span.get_text(strip=True))

        if area_span:
            area_text = clean(area_span.get_text(strip=True))

    return beds, baths, area_text


def parse_area(area_text):
    # Returns area as-is (e.g. "3.1 Marla", "5 Kanal") — no conversion
    return area_text


# -------------------------------------------------------
# Parse ALL feature sections
# -------------------------------------------------------
def parse_all_sections(soup: BeautifulSoup):
    out = {}
    try:
        ul = soup.find("ul", class_="_49fc0232")
        if not ul:
            return out
        for sec in ul.find_all("li", class_="_51519f00"):
            header = sec.find("div", class_="_359e9f89")
            title = clean(header.get_text(" ", strip=True)) if header else None
            if not title:
                continue
            items = []
            items_ul = sec.find("ul", class_="_3efd3392")
            if items_ul:
                for li in items_ul.find_all("li", class_="_59261156"):
                    span = li.find("span", class_="_9121cbf9")
                    if span:
                        txt = clean(span.get_text(" ", strip=True))
                        if txt:
                            items.append(txt)
            if items:
                out[title] = items
    except Exception:
        pass
    return out


def parse_rooms(section_map: dict):
    try:
        rooms_items = section_map.get("Rooms") or []
        room_details = {}
        other_rooms = []
        for it in rooms_items:
            if ":" in it:
                k, v = it.split(":", 1)
                room_details[clean(k)] = clean(v)
            else:
                other_rooms.append(it)
        return room_details, other_rooms
    except Exception:
        return {}, []


# -------------------------------------------------------
# 3) Property page -> full details
# -------------------------------------------------------
def scrape_property_details(prop_url: str, society_name: str, society_url: str):
    global all_feature_keys, first_fetch_printed

    try:
        soup = get_soup(prop_url)
        if soup is None:
            failed_properties.append({"url": prop_url, "error": "Failed to fetch"})
            return None

        # ✅ Beds, Baths, Area — from aria-label summary strip
        beds, baths, area_text = extract_beds_baths_area(soup)
        area = parse_area(area_text)  # e.g. "3.1 Marla", "5 Kanal"

        # Title
        title_tag  = soup.find(class_="cd230541")
        price_tag  = soup.find(class_="_2923a568")
        page_title = clean(title_tag.get_text(strip=True)) if title_tag else "Not listed"

        # Build display name: "2 BHK Flat for sale in Gulberg, Lahore"
        bhk_prefix = f"{beds} BHK " if beds else ""
        name = f"{bhk_prefix}Flat for sale in {society_name}, Lahore"

        # Price
        price_raw = clean(price_tag.get_text(strip=True)) if price_tag else ""
        if price_raw.startswith("PKR") and not price_raw.startswith("PKR "):
            price = "PKR " + price_raw[3:].strip()
        else:
            price = price_raw

        # Description
        desc_tag    = soup.find(class_="_3547dac9")
        description = clean(desc_tag.get_text(" ", strip=True)) if desc_tag else None

        # All feature sections
        section_map = parse_all_sections(soup)
        room_details, other_rooms = parse_rooms(section_map)

        # Nearby locations
        nearby_key       = "Nearby Locations and Other Facilities"
        nearby_locations = ", ".join(section_map.get(nearby_key, [])) or None

        # Merge all features (excluding Rooms section)
        merged_features = []
        for sec_title, items in section_map.items():
            if sec_title == "Rooms":
                continue
            merged_features.extend(items)

        features_str = " | ".join(merged_features) if merged_features else None

        # Dynamic features dict
        main_features = {}
        for t in merged_features:
            if ":" in t:
                k, v = t.split(":", 1)
                k, v = clean(k), clean(v)
                if k:
                    main_features[k] = v
            else:
                main_features[clean(t)] = True

        all_feature_keys.update(main_features.keys())

        floor_number = main_features.get("Floor") or main_features.get("Floor Number")
        total_floors = main_features.get("Floors in Building")
        built_year   = main_features.get("Built in year")

        # Property ID — last number in URL is the unique listing ID
        # e.g. /Property/some-title-189410-1907776.html → 1907776
        m = re.search(r"-(\d+)\.html$", prop_url)
        property_id = m.group(1) if m else "Not found"

        row = {
            "Property ID":    property_id,
            "Society":        society_name,
            "Society Link":   society_url,
            "Name":           name,
            "Page Title":     page_title,
            "Price":          price,
            "Area":           area,       # e.g. "3.1 Marla", "5 Kanal"
            "Bedrooms":       beds,
            "Baths":          baths,
            "Floor Number":   floor_number,
            "Total Floors":   total_floors,
            "Built Year":     built_year,
            "Address":        page_title,
            "Description":    description,
            "Features":       features_str,
            "Nearby Locations and Other Facilities": nearby_locations,
            "Rooms":          str(room_details),
            "Other Rooms":    ", ".join(other_rooms),
            "Link":           prop_url,
        }

        # Dynamic feature columns
        for k, v in main_features.items():
            row[k] = v

        # Section columns
        for sec_title, items in section_map.items():
            row[f"Section - {sec_title}"] = ", ".join(items)

        # ✅ Print first fetch details
        if not first_fetch_printed:
            print("\n" + "=" * 80)
            print("🔍 FIRST PROPERTY — DETAILED OUTPUT")
            print("=" * 80)
            print(f"  URL:          {prop_url}")
            print(f"  Society:      {society_name}")
            print(f"  Property ID:  {property_id}")
            print(f"  Name:         {name}")
            print(f"  Price:        {price}")
            print(f"  Bedrooms:     {beds}   ← from aria-label='Beds'")
            print(f"  Baths:        {baths}   ← from aria-label='Baths'")
            print(f"  Area:         {area}")
            print(f"  Floor:        {floor_number}  |  Total Floors: {total_floors}")
            print(f"  Built Year:   {built_year}")
            print(f"\n  Sections found ({len(section_map)}):")
            for sec_title, items in section_map.items():
                preview = ", ".join(items[:3])
                more = f" ... +{len(items)-3}" if len(items) > 3 else ""
                print(f"    [{sec_title}]: {preview}{more}")
            print(f"\n  Dynamic features: {len(main_features)}")
            print(f"  Total columns:    {len(row)}")
            print("=" * 80 + "\n")
            first_fetch_printed = True

        return row

    except Exception as e:
        error_msg = f"{type(e).__name__}: {str(e)}"
        failed_properties.append({"url": prop_url, "error": error_msg})
        return None


# -------------------------------------------------------
# 4) Auto-paginate through all pages of a society
# -------------------------------------------------------
def scrape_society_all_pages(society_name: str, society_url: str):
    page = 1
    last_fingerprint = None
    total_scraped = 0
    consecutive_failures = 0

    while True:
        page_url = make_page_url(society_url, page)
        print(f"   Page {page}: {page_url}")

        links = scrape_property_links(page_url)

        if not links:
            consecutive_failures += 1
            if consecutive_failures >= 2:
                print("   ✅ No listings for 2 consecutive pages — done with society.")
                break
            print("   ⚠️  No listings, trying next page...")
            page += 1
            continue
        else:
            consecutive_failures = 0

        # Stop if page repeats
        fingerprint = tuple(links[:25])
        if fingerprint == last_fingerprint:
            print("   ✅ Repeated listings detected — done with society.")
            break
        last_fingerprint = fingerprint

        print(f"   Found {len(links)} properties on page {page}")

        for idx, prop_url in enumerate(links, 1):
            # Use the full URL as dedup key — most reliable
            if prop_url in seen_property_ids:
                continue

            row = scrape_property_details(prop_url, society_name, society_url)
            if row:
                data.append(row)
                total_scraped += 1
                seen_property_ids.add(prop_url)

            if idx % 10 == 0:
                print(f"   Scraped {idx}/{len(links)} from page {page}  |  Total so far: {len(data)}")

        print(f"   ✅ Page {page} done. Society total: {total_scraped}")
        page += 1


# -------------------------------------------------------
# RUN
# -------------------------------------------------------
print("=" * 80)
print("ZAMEEN.COM — LAHORE FLATS SCRAPER")
print("=" * 80)
print(f"Master URL: {MASTER_URL}")
print("Rate limiting: 1-2s per request, 10s break every 50 requests\n")

societies = extract_societies(MASTER_URL)

if not societies:
    print("❌ No societies found. Exiting.")
    exit()

print(f"✅ Found {len(societies)} societies\n")
for i, s in enumerate(societies, 1):
    print(f"  {i:>3}. {s['society_name']}  ({s['society_count']} listings)")

print()

# -------------------------------------------------------
# Scrape each society
# -------------------------------------------------------
for i, s in enumerate(societies, 1):
    print(f"\n{'=' * 80}")
    print(f"[{i}/{len(societies)}]  {s['society_name']}  ({s['society_count']} properties)")
    print("=" * 80)

    scrape_society_all_pages(s["society_name"], s["society_url"])



# -------------------------------------------------------
# Final save
# -------------------------------------------------------
print(f"\n{'=' * 80}")
print("SCRAPING COMPLETE")
print("=" * 80)
print(f"✅ Total properties collected : {len(data)}")
print(f"❌ Failed properties          : {len(failed_properties)}")
print(f"📊 Total requests made        : {request_count}")

if len(data) == 0:
    print("\n⚠️  No data collected!")
    exit()

df = pd.DataFrame(data)

# Ensure all dynamic feature columns exist
for k in all_feature_keys:
    if k not in df.columns:
        df[k] = None

output_file = "lahore_flats_final.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"\n✅ Final CSV saved: {output_file}")
print(f"   Rows: {len(df)}  |  Columns: {len(df.columns)}")

# Save error log
if failed_properties or failed_pages:
    with open("flats_scraping_errors.json", "w") as f:
        json.dump({
            "failed_properties": failed_properties,
            "failed_pages": failed_pages
        }, f, indent=2)
    print(f"⚠️  Error log saved: flats_scraping_errors.json")

ZAMEEN.COM — LAHORE FLATS SCRAPER
Master URL: https://www.zameen.com/Flats_Apartments/Lahore-1-1.html
Rate limiting: 1-2s per request, 10s break every 50 requests

📄 Fetching master page: https://www.zameen.com/Flats_Apartments/Lahore-1-1.html
✅ Found 71 societies

    1. Askari  (956 listings)
    2. Bahria Town  (566 listings)
    3. Gulberg  (533 listings)
    4. Raiwind Road  (292 listings)
    5. DHA Defence  (251 listings)
    6. Bahria Orchard  (163 listings)
    7. Main Canal Bank Road  (89 listings)
    8. Johar Town  (60 listings)
    9. Defence Road  (34 listings)
   10. Land Breeze Housing Society  (31 listings)
   11. Model Town  (24 listings)
   12. Shanghai Road  (19 listings)
   13. Abdul Sattar Edhi Road  (17 listings)
   14. Khayaban-e-Amin  (16 listings)
   15. Bahria Nasheman  (16 listings)
   16. Izmir Town  (16 listings)
   17. Paragon City  (13 listings)
   18. Garden Town  (13 listings)
   19. Khayaban-e-Jinnah Road  (13 listings)
   20. Cooper Road  (13 listing